# VisDrone PrivacyRisk — Continuation từ Cell 6

Notebook này tiếp tục pipeline sau khi Cell 6 đã lưu kết quả detector/ByteTrack:
ByteTrack (chunked, auto-resume) → 25 đặc trưng (tinh gọn) → tấn công Re-ID → gán nhãn → XGBoost → export.

## Các sửa đổi quan trọng (bản này)

1. **Causal tuyệt đối**: sample tại `cutoff = window_start` chỉ dùng frame `<= cutoff`.
2. **Khớp runtime**: `FEATURE_AGG = 'last'` — lấy vector tại frame cuối `<= cutoff`,
   đúng bằng input của `step5_integrated_system.py`.
3. **Chuẩn hoá**: causal per-sequence z-score (Welford, `min_n=50`) riêng từng split.
4. **Cân bằng**: undersample âm còn 2:1 trong từng fold train (giữ mọi dương).
5. **Đánh giá**: GroupKFold theo sequence → OOF; val chọn tau; test hold-out.
6. **Calibration**: isotonic trên OOF (không fit trên tập train).
7. **Nhãn**: thêm margin/ambiguous/sample_weight để giảm ảnh hưởng nhãn nhiễu.

## Đặc trưng schema v2 (25 đặc trưng)
- Loại 4 đặc trưng gần như hằng/trùng lặp: `track_state`, `aspect_dev`, `is_trunc`, `frame_det_delta`.
- Thêm 8 đặc trưng causal: `conf_mean5`, `conf_slope5`, `area_norm`, `height_norm`, `center_y_norm`, `iou_prev`, `scale_change`, `crop_contrast`.
- 4 đặc trưng bị loại vẫn được tính và báo univariate AUC để audit; permutation importance theo nhóm được xuất trong `risk_metrics.json`.


In [ ]:
# Kaggle bootstrap: keep notebook runnable when helper .py files were not uploaded.
import json
from pathlib import Path
import numpy as np
import pandas as pd
import shutil
FEATURES = ("conf", "conf_std5", "conf_delta", "conf_mean5", "conf_slope5", "lap_var_log", "brightness", "motion_blur_ratio", "crop_contrast", "track_age", "hit_ratio", "time_since_update", "speed_norm", "pred_uncert", "iou_prev", "scale_change", "min_side_px", "area_norm", "height_norm", "center_y_norm", "dist_edge", "nearest_dist_norm", "n_neighbors_r2_norm", "overlap_max", "n_det_expected_ratio")
def add_validation_split(index, seed=42, validation_fraction=0.20):
    result = index.copy()
    if "val" in set(result["split"].astype(str)):
        return result
    sequences = sorted(result.loc[result["split"].eq("train"), "sequence"].unique())
    if len(sequences) < 2:
        raise ValueError("at least two train sequences are required")
    rng = np.random.default_rng(seed); rng.shuffle(sequences)
    chosen = set(sequences[:max(1, int(round(len(sequences) * validation_fraction)))])
    result.loc[result["split"].eq("train") & result["sequence"].isin(chosen), "split"] = "val"
    return result
def repair_tracker_features(df):
    result = df.copy()
    result["time_since_update"] = pd.to_numeric(result["time_since_update"], errors="coerce").clip(lower=0).fillna(0)
    result["hit_ratio"] = pd.to_numeric(result["hit_ratio"], errors="coerce").clip(0, 1)
    result["track_state"] = pd.to_numeric(result["track_state"], errors="coerce").fillna(2).clip(0, 2)
    return result
def save_model_metadata(model_path, feature_names=FEATURES):
    if tuple(feature_names) != tuple(FEATURES): raise ValueError("feature order mismatch")
    path = Path(model_path).with_suffix(Path(model_path).suffix + ".metadata.json")
    path.write_text(json.dumps({"feature_names": list(FEATURES), "feature_count": len(FEATURES)}, indent=2), encoding="utf-8")
    return path
def load_model_metadata(model_path):
    path = Path(model_path).with_suffix(Path(model_path).suffix + ".metadata.json")
    data = json.loads(path.read_text(encoding="utf-8"))
    if tuple(data.get("feature_names", ())) != tuple(FEATURES): raise ValueError("feature metadata mismatch")
    return tuple(FEATURES)


In [ ]:
# â”€â”€ CELL 1/13 Â· Setup + kiá»ƒm tra pháº§n cá»©ng â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Chá»‰ cÃ i gÃ³i cÃ²n THIáº¾U. KhÃ´ng dÃ¹ng -U Ä‘á»ƒ trÃ¡nh pip gá»¡ báº£n cÃ³ sáºµn rá»“i táº£i lá»—i
# (Kaggle: pip upgrade = uninstall trÆ°á»›c, máº¥t máº¡ng giá»¯a chá»«ng lÃ  máº¥t luÃ´n gÃ³i).

import importlib.util
import subprocess
import sys

REQUIRED = {
    'ultralytics': 'ultralytics',
    'xgboost': 'xgboost',
    'shap': 'shap',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'yaml': 'pyyaml',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'matplotlib': 'matplotlib',
    'cv2': 'opencv-python-headless',
    'torch': 'torch',
}

_missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print('gÃ³i cÃ²n thiáº¿u:', _missing)
    _r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError(
            'pip install tháº¥t báº¡i. Báº­t Kaggle Settings â†’ Internet: On rá»“i cháº¡y láº¡i cell nÃ y.\n'
            + (_r.stderr or '')[-1500:]
        )
else:
    print('táº¥t cáº£ gÃ³i Ä‘Ã£ cÃ³ sáºµn, khÃ´ng cáº§n cÃ i.')

# torchreid lÃ  tuá»³ chá»n (Re-ID dÃ¹ng OSNet). Thiáº¿u thÃ¬ cell 9 tá»± fallback ResNet50.
if importlib.util.find_spec('torchreid') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchreid'],
                   capture_output=True, text=True)

import os
for _k in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_k] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'

import csv
import ctypes
import gc
import json
import math
import random
import re
import shutil
import sys
import time
import warnings
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO

# glibc (Kaggle = Linux): giá»›i háº¡n arena + cho phÃ©p tráº£ heap Ä‘Ã£ free vá» OS.
# DÃ¹ng kÃ¨m malloc_trim(0) trong Cell 7 Ä‘á»ƒ RSS khÃ´ng phÃ¬nh dáº§n.
_LIBC = None
try:
    _LIBC = ctypes.CDLL('libc.so.6')
    _LIBC.mallopt(-8, 2)   # M_ARENA_MAX = 2
except Exception:
    _LIBC = None

warnings.filterwarnings('ignore')
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = True

print('python     :', sys.version.split()[0])
print('ultralytics:', ultralytics.__version__)
print('torch      :', torch.__version__, '| cuda:', torch.cuda.is_available())

n_gpu = torch.cuda.device_count()
print('GPU count  :', n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} | {p.total_memory / 2**30:.1f} GiB')

try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'RAM        : {vm.total / 2**30:.1f} GiB total | {vm.available / 2**30:.1f} GiB available')
except ImportError:
    pass

for _p in ('/kaggle/working', '/tmp', '/kaggle/input'):
    if os.path.isdir(_p):
        u = shutil.disk_usage(_p)
        print(f'disk {_p:<16}: {u.free / 2**30:.1f} GiB free / {u.total / 2**30:.1f} GiB')

if n_gpu < 2:
    print('Cáº¢NH BÃO: chá»‰ tháº¥y', n_gpu, 'GPU. Váº«n cháº¡y Ä‘Æ°á»£c nhÆ°ng cháº­m hÆ¡n.')
assert torch.cuda.is_available(), 'Báº­t GPU T4 x2 trong Kaggle Settings trÆ°á»›c khi cháº¡y.'


In [ ]:
# â”€â”€ CELL 2/13 Â· CONFIG â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Chá»‰ sá»­a cell nÃ y.

from pathlib import Path

# ===== Dá»® LIá»†U =====
# CÃ¡ch 1: 1 thÆ° má»¥c gá»‘c, notebook tá»± chia sequence theo SPLIT_RATIOS.
DATA_ROOT = Path('/kaggle/input/visdrone-vid')

# CÃ¡ch 2 (Ä‘ang dÃ¹ng): 3 thÆ° má»¥c riÃªng cho train/val/test.
USE_SEPARATE_DIRS = True
TRAIN_DIR = Path('/kaggle/input/datasets/mtu2006/visdrone-train/VisDrone2019-VID-train')
VAL_DIR   = Path('/kaggle/input/datasets/mtu2006/visdrone-val/VisDrone2019-VID-val')
TEST_DIR  = Path('/kaggle/input/datasets/mtu2006/visdrone-test/VisDrone2019-VID-test-dev')

SPLIT_RATIOS = (4, 1, 2)
SPLIT_SEED = 42

# ===== THÆ¯ Má»¤C =====
WORK = Path('/kaggle/working')
BUILD_DIR = WORK / 'privacy_build'
RUNS_DIR = WORK / 'runs'
ARTIFACT = WORK / 'artifacts'

SEQ_INDEX = WORK / 'sequence_index.csv'
GT_CSV = WORK / 'gt_boxes.csv'
DATA_YAML = WORK / 'data_person.yaml'
LAYOUT_JSON = WORK / 'layout.json'

TRACKS_CSV = ARTIFACT / 'tracks_frame.csv'
FEATURES_CSV = ARTIFACT / 'features_frame.csv'
SAMPLES_CSV = ARTIFACT / 'samples.csv'
ATTACK_CSV = ARTIFACT / 'attack_results.csv'
TAU_JSON = ARTIFACT / 'tau.json'
RISK_CSV = ARTIFACT / 'risk_dataset.csv'
SCHEMA_JSON = ARTIFACT / 'feature_schema.json'
RISK_MODEL_JSON = ARTIFACT / 'risk_model.json'
RISK_OOF_CSV = ARTIFACT / 'risk_oof.csv'
RISK_METRICS_JSON = ARTIFACT / 'risk_metrics.json'
DET_METRICS_JSON = ARTIFACT / 'det_metrics.json'

# ===== DETECTOR =====
MODEL_NAME = 'yolo26n.pt'
FALLBACK = ['yolo11s.pt', 'yolo26n.pt', 'yolo11n.pt']
PERSON_CATS = (1, 2)            # VisDrone: 1=pedestrian, 2=people
IMGSZ = 1024
BATCH = 32
EPOCHS = 30
WORKERS = 4
DEVICE = '0'
DEVICE_EVAL = 0
SEED = 42
PATIENCE = 15
RUN_NAME = 'yolo26s_visdrone_recall_t4x2'
MAX_DET = 1000

AUG = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5.0, translate=0.15, scale=0.7, shear=2.0, perspective=0.0005,
    fliplr=0.5, flipud=0.0,
    mosaic=0.8, mixup=0.05, copy_paste=0.0, erasing=0.2,
    close_mosaic=12, multi_scale=0.0,
)

# ===== ÄÃNH GIÃ DETECTOR =====
VAL_CONF = 0.001
VAL_IOU = 0.7
SWEEP_MAX_IMAGES = 1200
CONF_GRID = (0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50)
IOU_GRID = (0.5, 0.6, 0.7)
MOT_IOU_TH = 0.5

# ===== TRACKING =====
TRACK_SPLITS = ('train', 'val', 'test')
TRACK_CONF = 0.15
TRACK_IOU = 0.5
TRACK_BUFFER = 45
TRACK_ID_IOU = 0.4
TRACK_RESUME = True      # bá» qua sequence Ä‘Ã£ track xong trong *.partial.csv

# ===== Tá»° PHá»¤C Há»’I KHI Gáº¦N Háº¾T RAM =====
TRACK_RAM_LIMIT_GB = None    # None = tá»± dÃ² (cgroup Kaggle); Ä‘áº·t sá»‘ Ä‘á»ƒ Ã©p ngÆ°á»¡ng
TRACK_RAM_STOP_FRAC = 0.85   # tá»± dá»n model khi dÃ¹ng > 85% giá»›i háº¡n RAM
TRACK_MEM_CHECK_EVERY = 25   # kiá»ƒm tra RAM má»—i N frame trong 1 sequence
TRACK_AUTO_RESTART_MAX = 6   # sá»‘ láº§n tá»± dá»n + cháº¡y tiáº¿p trÆ°á»›c khi dá»«ng háº³n

# ===== ĐẶC TRƯNG (schema v2) =====
# 25 đặc trưng chính được đưa vào XGBoost.
# 4 đặc trưng cũ bị loại khỏi model nhưng vẫn được tính để cell metrics
# so sánh univariate AUC (chứng minh việc loại bỏ là hợp lý).
FEATURE_SCHEMA_VERSION = 2

FEATURES = [
    # chất lượng detection / appearance
    'conf', 'conf_std5', 'conf_delta', 'conf_mean5', 'conf_slope5',
    'lap_var_log', 'brightness', 'motion_blur_ratio', 'crop_contrast',
    # độ ổn định track theo thời gian
    'track_age', 'hit_ratio', 'time_since_update',
    'speed_norm', 'pred_uncert', 'iou_prev', 'scale_change',
    # hình học / phối cảnh
    'min_side_px', 'area_norm', 'height_norm', 'center_y_norm', 'dist_edge',
    # bối cảnh đám đông / detection
    'nearest_dist_norm', 'n_neighbors_r2_norm', 'overlap_max',
    'n_det_expected_ratio',
]

DROPPED_FEATURES = ['track_state', 'aspect_dev', 'is_trunc', 'frame_det_delta']

FEATURE_DROP_RATIONALE = {
    'track_state': 'Gần như luôn = 1 (tracked) với output ByteTrack -> gần như hằng số.',
    'aspect_dev': 'Nhiễu với box nhỏ; tương quan cao với min_side_px/area_norm.',
    'is_trunc': 'Chỉ là ngưỡng hoá của dist_edge, làm mất thông tin liên tục.',
    'frame_det_delta': 'Delta 1 frame rất nhiễu; còn phụ thuộc lịch phát hiện của scheduler.',
}

FEATURE_GROUPS = {
    'detection_quality': [
        'conf', 'conf_std5', 'conf_delta', 'conf_mean5', 'conf_slope5',
        'lap_var_log', 'brightness', 'motion_blur_ratio', 'crop_contrast',
    ],
    'track_stability': [
        'track_age', 'hit_ratio', 'time_since_update',
        'speed_norm', 'pred_uncert', 'iou_prev', 'scale_change',
    ],
    'geometry': [
        'min_side_px', 'area_norm', 'height_norm', 'center_y_norm', 'dist_edge',
    ],
    'crowd_context': [
        'nearest_dist_norm', 'n_neighbors_r2_norm', 'overlap_max',
        'n_det_expected_ratio',
    ],
}

DIAGNOSTIC_FEATURES = list(DROPPED_FEATURES)
CONF_WINDOW = 5
PRED_UNCERT_WINDOW = 5
NEIGHBOR_RADIUS_FRAC = 0.05   # đồng bộ step5 runtime
NEIGHBOR_K = 5                # đồng bộ step5 runtime
EWMA_ALPHA = 0.3
TRUNC_MARGIN_PX = 2
CROP_RESIZE = 224             # đồng bộ step5 runtime
FRAME_STRIDE = 1

assert len(FEATURES) == 25, 'Phải có đúng 25 đặc trưng model'
assert len(set(FEATURES)) == len(FEATURES), 'Đặc trưng bị trùng'
assert set(DROPPED_FEATURES).isdisjoint(FEATURES), 'Đặc trưng bị loại vẫn nằm trong model'
assert sorted(sum(FEATURE_GROUPS.values(), [])) == sorted(FEATURES), 'FEATURE_GROUPS không khớp FEATURES'

# ===== Ghi đè đường dẫn: ưu tiên file đã giải nén từ final.zip (nếu có) =====
_override_model = WORK / 'artifacts' / 'yolo26s.pt'
_override_seq_index = WORK / 'sequence_index.csv'
_override_data_yaml = WORK / 'data_person.yaml'

BEST_MODEL = None
SEQ_INDEX = WORK / 'sequence_index.csv'
DATA_YAML = WORK / 'data_person.yaml'

if _override_model.is_file():
    BEST_MODEL = _override_model
    print('Sử dụng model:', BEST_MODEL)
else:
    print('Không tìm thấy model tại', _override_model, '- giữ cấu hình mặc định')

if _override_seq_index.is_file():
    SEQ_INDEX = _override_seq_index
    print('Sử dụng sequence_index:', SEQ_INDEX)
else:
    print('Không tìm thấy sequence_index tại', _override_seq_index, '- giữ cấu hình mặc định')

if _override_data_yaml.is_file():
    DATA_YAML = _override_data_yaml
    print('Sử dụng data_yaml:', DATA_YAML)
else:
    print('Không tìm thấy data_yaml tại', _override_data_yaml, '- giữ cấu hình mặc định')

# ===== Cá»¬A Sá»” MáºªU =====
WINDOW_SIZE = 60          # 0 = dÃ¹ng cáº£ track
WINDOW_STRIDE = 1        # sliding window: dá»± Ä‘oÃ¡n á»Ÿ tá»«ng frame
MIN_WINDOW_FRAMES = 20

# ===== áº¨N DANH =====
ANON_METHOD = 'blur'      # 'blur' | 'pixelate'
ANON_KSIZE = 15
ANON_PIXEL_BLOCK = 8

# ===== RE-ID =====
REID_BACKBONE = 'auto'    # 'auto' | 'osnet_x1_0' | 'resnet50'
REID_INPUT = 256
REID_BATCH = 64
EMBED_STRIDE = 5
PROBE_STRIDE = 10
MAX_PROBE_CROPS = 8
MAX_GALLERY_IDS = 200
GALLERY_MIN_GAP = 30      # tÄƒng sá»‘ máº«u cÃ³ gallery nhÆ°ng váº«n trÃ¡nh cá»­a sá»• lÃ¢n cáº­n
GALLERY_CROSS_SEQUENCE = False
MIN_GALLERY_IDS = 2

# ===== GÃN NHÃƒN =====
TAU_SELECT_SPLIT = 'val'
LABEL_MIN_MATCH_RATE = 0.25

# ===== MÔ HÌNH RỦI RO =====
# Cấu hình cố ý nhỏ hơn bản cũ: với 25 đặc trưng tín hiệu yếu, cây càng sâu
# càng dễ học thuộc sequence thay vì học quy luật chung.
XGB_PARAMS = dict(
    n_estimators=600, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
    reg_lambda=5.0, reg_alpha=0.5, objective='binary:logistic',
    eval_metric='aucpr', tree_method='hist', early_stopping_rounds=50,
    n_jobs=4, random_state=SEED,
    scale_pos_weight=5.0,
)
N_FOLDS = 5
RISK_THRESHOLD = 0.5

# ===== CÂN BẰNG / CHUẨN HOÁ / GIAO THỨC ĐÁNH GIÁ =====
NEG_POS_RATIO = 1.5          # undersample âm trong từng fold để đạt âm:dương = 1.5:1
FEATURE_AGG = 'last'         # 'last' = khớp runtime (frame tại cutoff); 'window_mean' = trung bình cửa sổ
NORM_MIN_SAMPLES = 50        # số vector thô đầu tiên trước khi bật z-score (khớp step5)
LABEL_TAU_MARGIN = 0.02      # vùng mờ quanh tau -> hạ sample_weight / loại khỏi train
DROP_AMBIGUOUS = True        # loại mẫu nhãn mờ khỏi TRAIN (vẫn giữ để đánh giá val/test)
TARGET_FPR = 0.10            # tau vận hành chọn theo FPR mục tiêu trên val
STRICT_SEQ_DISJOINT = True   # raise nếu train/val/test trùng sequence
GALLERY_MAX_PER_ID = 200     # số embedding tối đa mỗi identity trong gallery Re-ID

RISK_TAU_JSON = ARTIFACT / 'risk_tau.json'
RISK_NORM_JSON = ARTIFACT / 'risk_norm_stats.json'
RISK_SPLIT_METRICS_JSON = ARTIFACT / 'risk_split_metrics.json'

# ===== EXPORT =====
EXPORT_ONNX = True
ONNX_OPSET = 17

for _p in (WORK, RUNS_DIR, ARTIFACT, BUILD_DIR):
    _p.mkdir(parents=True, exist_ok=True)

print('CONFIG OK |', MODEL_NAME, '| imgsz', IMGSZ, '| batch', BATCH, '| epochs', EPOCHS)
if USE_SEPARATE_DIRS:
    for _k, _v in (('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)):
        print(f'data {_k:<5}:', _v, '| exists:', _v.is_dir())
else:
    print('data  :', DATA_ROOT, '| exists:', DATA_ROOT.is_dir())
print('features:', len(FEATURES), '| neg:pos =', NEG_POS_RATIO, '| feature_agg =', FEATURE_AGG)


In [ ]:
# â”€â”€ CELL 3/13 Â· Chuáº©n hoÃ¡ dataset â†’ YOLO person â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# VÃ o : DATA_ROOT (hoáº·c TRAIN_DIR/VAL_DIR/TEST_DIR)
# Ra  : BUILD_DIR/<split>/images+labels, data_person.yaml,
#       sequence_index.csv, gt_boxes.csv, layout.json

import yaml
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
PSEUDO_MAX_GAP = 5
PSEUDO_IOU = 0.3
SPLITS = ('train', 'val', 'test')


def xyxy_iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)


def find_layout(path):
    if not path.is_dir():
        return None, None
    queue = [(path, 0)]
    while queue:
        cur, depth = queue.pop(0)
        if (cur / 'images').is_dir() and (cur / 'labels').is_dir():
            return 'yolo', cur
        if (cur / 'annotations').is_dir() and (cur / 'sequences').is_dir():
            return 'raw', cur
        if depth < 2:
            queue += [(p, depth + 1) for p in sorted(cur.iterdir()) if p.is_dir()]
    return None, None


def image_size(path):
    try:
        with Image.open(path) as im:
            return im.size
    except Exception:
        arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        return (arr.shape[1], arr.shape[0]) if arr is not None else (0, 0)


def parse_annotation(path):
    rows = defaultdict(list)
    for raw in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        f = raw.split(',')
        if len(f) != 10:
            continue
        try:
            frame, tid = int(float(f[0])), int(float(f[1]))
            x, y, w, h = (float(v) for v in f[2:6])
            score, cat = int(float(f[6])), int(float(f[7]))
        except ValueError:
            continue
        if score == 0 or not (1 <= cat <= 11) or w <= 0 or h <= 0:
            continue
        rows[frame].append((cat, x, y, w, h, tid))
    return rows


def copy_many(pairs, workers=16):
    # Kaggle: /kaggle/input lÃ  mount FUSE (~1 MB/s/file, overhead má»Ÿ file ráº¥t cao).
    # Symlink váº«n Ä‘á»c qua mount Ä‘Ã³ nÃªn pháº£i copy tháº­t vá» /kaggle/working.
    # Copy song song Ä‘á»ƒ bÃ¹ chi phÃ­ má»Ÿ file.
    todo = [(s, d) for s, d in pairs if not d.exists()]
    if not todo:
        return 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(lambda p: shutil.copy2(p[0], p[1]), todo))
    return len(todo)


def write_label(path, lines):
    path.write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')


def parse_name(path):
    m = re.match(r'^(.*)_frame(\d+)$', path.stem)
    if m:
        return m.group(1), int(m.group(2))
    m = re.match(r'^(.+?)_(\d+)$', path.stem)
    if m:
        return m.group(1), int(m.group(2))
    return (path.parent.name or '__all__'), 0


def collect_sequences(kind, root):
    if kind == 'raw':
        return sorted(p.name for p in (root / 'sequences').iterdir() if p.is_dir())
    seqs = set()
    for p in (root / 'images').rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            seqs.add(parse_name(p)[0])
    return sorted(seqs)


def split_sequences(seqs, ratios, seed):
    arr = sorted(seqs)
    random.Random(seed).shuffle(arr)
    n = len(arr)
    a, b, c = ratios
    total = a + b + c
    n_tr = max(1, int(round(n * a / total)))
    n_va = max(1, int(round(n * b / total)))
    if n_tr + n_va >= n:
        n_tr = max(1, n - 2)
        n_va = 1
    return {'train': arr[:n_tr], 'val': arr[n_tr:n_tr + n_va], 'test': arr[n_tr + n_va:]}


def convert_raw(root, split, seq_filter, out, index_rows, gt_rows):
    img_dir, lbl_dir = out / 'images', out / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    ann = {p.stem: parse_annotation(p) for p in (root / 'annotations').glob('*.txt')}
    seqs = [p for p in sorted((root / 'sequences').iterdir()) if p.is_dir()]
    if seq_filter is not None:
        seqs = [p for p in seqs if p.name in seq_filter]

    n_img = 0
    pairs = []
    for seq in seqs:
        files = sorted(p for p in seq.iterdir() if p.suffix.lower() in IMG_EXTS)
        if not files:
            continue
        iw, ih = image_size(files[0])
        rows = ann.get(seq.name, {})
        for src in files:
            frame = int(src.stem) if src.stem.isdigit() else n_img
            flat = f'{seq.name}__{frame:07d}.jpg'
            pairs.append((src, img_dir / flat))
            lines = []
            for cat, x, y, w, h, tid in rows.get(frame, []):
                if cat not in PERSON_CATS:
                    continue
                lines.append(f'0 {(x + w / 2) / iw:.6f} {(y + h / 2) / ih:.6f} {w / iw:.6f} {h / ih:.6f}')
                gt_rows.append({'split': split, 'sequence': seq.name, 'frame': frame,
                                'target_id': int(tid), 'x1': x, 'y1': y,
                                'x2': x + w, 'y2': y + h, 'w': w, 'h': h})
            write_label(lbl_dir / f'{flat[:-4]}.txt', lines)
            index_rows.append({'split': split, 'sequence': seq.name, 'frame': frame,
                               'flat_name': flat, 'source_image': str(src),
                               'width': iw, 'height': ih})
            n_img += 1
    copy_many(pairs)
    return n_img


def convert_yolo(root, split, seq_filter, out, index_rows, gt_rows):
    img_dir, lbl_dir = out / 'images', out / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    sizes = {}
    n_img = 0
    pairs = []
    for src in sorted((root / 'images').rglob('*')):
        if src.suffix.lower() not in IMG_EXTS:
            continue
        seq, frame = parse_name(src)
        if seq_filter is not None and seq not in seq_filter:
            continue
        if seq not in sizes:
            sizes[seq] = image_size(src)
        iw, ih = sizes[seq]

        rel = src.relative_to(root / 'images')
        lab = (root / 'labels' / rel).with_suffix('.txt')
        lines = []
        if lab.is_file():
            for ln in lab.read_text(encoding='utf-8', errors='ignore').splitlines():
                f = ln.split()
                if len(f) != 5:
                    continue
                try:
                    cls = int(float(f[0]))
                    cx, cy, w, h = (float(v) for v in f[1:])
                except ValueError:
                    continue
                if cls not in (0, 1):
                    continue
                x1, y1 = (cx - w / 2) * iw, (cy - h / 2) * ih
                x2, y2 = (cx + w / 2) * iw, (cy + h / 2) * ih
                lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
                gt_rows.append({'split': split, 'sequence': seq, 'frame': frame,
                                'target_id': -1, 'x1': x1, 'y1': y1,
                                'x2': x2, 'y2': y2, 'w': x2 - x1, 'h': y2 - y1})

        flat = f'{seq}__{frame:07d}.jpg'
        pairs.append((src, img_dir / flat))
        write_label(lbl_dir / f'{flat[:-4]}.txt', lines)
        index_rows.append({'split': split, 'sequence': seq, 'frame': frame,
                           'flat_name': flat, 'source_image': str(src),
                           'width': iw, 'height': ih})
        n_img += 1
    copy_many(pairs)
    return n_img


def assign_pseudo_ids(gt_rows):
    groups = defaultdict(list)
    for i, r in enumerate(gt_rows):
        if r['target_id'] < 0:
            groups[(r['split'], r['sequence'])].append(i)

    for key, idxs in groups.items():
        idxs.sort(key=lambda i: gt_rows[i]['frame'])
        by_frame = defaultdict(list)
        for i in idxs:
            by_frame[gt_rows[i]['frame']].append(i)
        active, next_id = {}, 0
        for frame in sorted(by_frame):
            for tid in list(active):
                if frame - active[tid][0] > PSEUDO_MAX_GAP:
                    del active[tid]
            cur = by_frame[frame]
            boxes = np.array([[gt_rows[i][k] for k in ('x1', 'y1', 'x2', 'y2')] for i in cur], float)
            used, used_j = set(), set()
            if active:
                tids = list(active)
                gb = np.array([active[t][1] for t in tids], float)
                iou = xyxy_iou(boxes, gb)
                for flat in np.argsort(-iou, axis=None):
                    a, b = divmod(int(flat), iou.shape[1])
                    if iou[a, b] < PSEUDO_IOU:
                        break
                    if a in used or b in used_j:
                        continue
                    used.add(a)
                    used_j.add(b)
                    gt_rows[cur[a]]['target_id'] = tids[b]
                    active[tids[b]] = (frame, boxes[a])
            for a, i in enumerate(cur):
                if a in used:
                    continue
                gt_rows[i]['target_id'] = next_id
                active[next_id] = (frame, boxes[a])
                next_id += 1
    return sum(1 for r in gt_rows if r['target_id'] < 0)



if BUILD_DIR.joinpath('train', 'images').is_dir() and GT_CSV.is_file():
    print('?? c? privacy_build v? gt_boxes.csv; b? qua chu?n h?a dataset.')
else:
    index_rows, gt_rows = [], []
    layout_info = {'mode': 'separate' if USE_SEPARATE_DIRS else 'auto_split', 'sources': {}}

    _free_gb = shutil.disk_usage(WORK).free / 2 ** 30
    print(f'Ä‘Ä©a trá»‘ng /kaggle/working: {_free_gb:.1f} GB (cáº§n ~12 GB Ä‘á»ƒ copy áº£nh)')
    if _free_gb < 12:
        print('Cáº¢NH BÃO: thiáº¿u dung lÆ°á»£ng. Dá»n /kaggle/working hoáº·c giáº£m dá»¯ liá»‡u.')

    if USE_SEPARATE_DIRS:
        dirs = {'train': TRAIN_DIR, 'val': VAL_DIR, 'test': TEST_DIR}
        for split in SPLITS:
            kind, root = find_layout(dirs[split])
            if kind is None:
                raise FileNotFoundError(f'{split}: khÃ´ng tháº¥y dá»¯ liá»‡u trong {dirs[split]}')
            layout_info['sources'][split] = {'kind': kind, 'root': str(root)}
            out = BUILD_DIR / split
            if kind == 'raw':
                n = convert_raw(root, split, None, out, index_rows, gt_rows)
            else:
                n = convert_yolo(root, split, None, out, index_rows, gt_rows)
            print(f'{split:<5} {n:>5} áº£nh ({kind})')
    else:
        src = DATA_ROOT
        kind, root = find_layout(src)
        if kind is None:
            raise FileNotFoundError(
                f'KhÃ´ng tháº¥y annotations/+sequences/ hoáº·c images/+labels/ trong {src}. '
                f'Kiá»ƒm tra láº¡i DATA_ROOT hoáº·c báº­t USE_SEPARATE_DIRS.'
            )
        all_seqs = collect_sequences(kind, root)
        if len(all_seqs) < 3:
            raise ValueError(f'Chá»‰ cÃ³ {len(all_seqs)} sequence, cáº§n >=3 Ä‘á»ƒ tá»± chia. '
                             f'Äáº·t USE_SEPARATE_DIRS = True náº¿u Ä‘Ã£ cÃ³ 3 thÆ° má»¥c riÃªng.')
        seqs = split_sequences(all_seqs, SPLIT_RATIOS, SPLIT_SEED)
        layout_info.update({'kind': kind, 'root': str(root), 'sequences': seqs})
        print('auto split:', {k: len(v) for k, v in seqs.items()})
        for split in SPLITS:
            out = BUILD_DIR / split
            if kind == 'raw':
                n = convert_raw(root, split, set(seqs[split]), out, index_rows, gt_rows)
            else:
                n = convert_yolo(root, split, set(seqs[split]), out, index_rows, gt_rows)
            print(f'{split:<5} {n:>5} áº£nh | {seqs[split]}')

    n_pseudo = sum(1 for r in gt_rows if r['target_id'] < 0)
    assign_pseudo_ids(gt_rows)
    layout_info['pseudo_identity'] = n_pseudo > 0
    layout_info['pseudo_rows'] = n_pseudo

    df_idx = pd.DataFrame(index_rows).sort_values(['split', 'sequence', 'frame'])
    df_gt = pd.DataFrame(gt_rows).sort_values(['split', 'sequence', 'frame', 'target_id'])
    df_idx.to_csv(SEQ_INDEX, index=False)
    df_gt.to_csv(GT_CSV, index=False)
    LAYOUT_JSON.write_text(json.dumps(layout_info, indent=2, ensure_ascii=False), encoding='utf-8')

    DATA_YAML.write_text(yaml.safe_dump({
        'path': str(WORK),
        'train': str(BUILD_DIR / 'train' / 'images'),
        'val': str(BUILD_DIR / 'val' / 'images'),
        'test': str(BUILD_DIR / 'test' / 'images'),
        'nc': 1,
        'names': {0: 'person'},
    }, sort_keys=False, allow_unicode=True), encoding='utf-8')

    print('\nsequence_index.csv:', len(df_idx), 'dÃ²ng')
    print('gt_boxes.csv      :', len(df_gt), 'box |', df_gt.target_id.nunique(), 'identity')
    if n_pseudo:
        print(f'Cáº¢NH BÃO: {n_pseudo} box khÃ´ng cÃ³ target_id â†’ dÃ¹ng pseudo-ID (xáº¥p xá»‰).')
    print(DATA_YAML.read_text(encoding='utf-8'))


In [ ]:
# ?? CELL 6-RESTORE ? Kh?i ph?c k?t qu? v? model t? Cell 6 ?????
# Ch?y cell n?y sau setup/config, tr??c Cell 7.

WORK = Path('/kaggle/working')
ARTIFACT = WORK / 'artifacts'
RUNS_DIR = WORK / 'runs'
ARTIFACT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = ARTIFACT / 'cell6_checkpoint'
CHECKPOINT_FILE = CHECKPOINT_DIR / 'detector_checkpoint.json'
DET_METRICS_JSON = ARTIFACT / 'det_metrics.json'

_checkpoint = {}
if CHECKPOINT_FILE.is_file():
    _checkpoint = json.loads(CHECKPOINT_FILE.read_text(encoding='utf-8'))
elif DET_METRICS_JSON.is_file():
    _checkpoint = {'det_metrics': json.loads(DET_METRICS_JSON.read_text(encoding='utf-8'))}

_threshold = _checkpoint.get('det_metrics', {}).get('best_threshold', {})
CONF_BEST = float(_checkpoint.get('CONF_BEST', _threshold.get('conf', TRACK_CONF)))
IOU_BEST = float(_checkpoint.get('IOU_BEST', _threshold.get('iou', TRACK_IOU)))
IMGSZ_USED = int(_checkpoint.get('IMGSZ_USED', IMGSZ))
BATCH_USED = int(_checkpoint.get('BATCH_USED', BATCH))

# Tá»± tÃ¬m ZIP/model Ä‘Ã£ upload trong /kaggle/input hoáº·c /kaggle/working.
# ZIP chá»‰ Ä‘Æ°á»£c giáº£i nÃ©n náº¿u cÃ³ yolo26s.pt, trÃ¡nh giáº£i nÃ©n nháº§m dataset lá»›n.
_search_roots = [WORK, Path('/kaggle/input')]
_zip_candidates = []
for _root in _search_roots:
    if _root.is_dir():
        _zip_candidates.extend(_root.rglob('*.zip'))
for _zip_path in _zip_candidates:
    try:
        with zipfile.ZipFile(_zip_path) as _zf:
            _has_model = any(Path(n).name == 'yolo26s.pt' for n in _zf.namelist())
        if _has_model:
            print('Giáº£i nÃ©n model ZIP:', _zip_path)
            with zipfile.ZipFile(_zip_path) as _zf:
                _zf.extractall(WORK)
    except (OSError, zipfile.BadZipFile) as _err:
        print('Bá» qua ZIP khÃ´ng Ä‘á»c Ä‘Æ°á»£c:', _zip_path, '|', str(_err)[:120])

_model_candidates = []
for _root in _search_roots:
    if _root.is_dir():
        _model_candidates.extend(_root.rglob('yolo26s.pt'))
_model_candidates = [p for p in _model_candidates if p.is_file()]
if _model_candidates:
    _preferred = [p for p in _model_candidates if 'weights' in p.parts]
    BEST_MODEL = (_preferred or _model_candidates)[0]
else:
    BEST_MODEL = RUNS_DIR / RUN_NAME / 'weights' / 'yolo26s.pt'
assert BEST_MODEL.is_file(), (
    'KhÃ´ng tÃ¬m tháº¥y yolo26s.pt. HÃ£y upload ZIP káº¿t quáº£ cÃ³ runs/*/weights/yolo26s.pt '
    'vÃ o Kaggle Input hoáº·c /kaggle/working.'
)
# Äá»“ng bá»™ cÃ¡c file pipeline náº¿u chÃºng náº±m trong thÆ° má»¥c giáº£i nÃ©n khÃ¡c.
for _name in ('sequence_index.csv', 'data_person.yaml', 'gt_boxes.csv'):
    _current = globals().get(_name.upper().replace('.CSV', '').replace('.YAML', ''), None)
    _found = next((p for p in WORK.rglob(_name) if p.is_file()), None)
    if _found is not None:
        if _name == 'sequence_index.csv': SEQ_INDEX = _found
        elif _name == 'data_person.yaml': DATA_YAML = _found
        else: GT_CSV = _found

for _required in (SEQ_INDEX, DATA_YAML):
    assert _required.is_file(), f'Thi?u file ??u v?o: {_required}'

print('?? kh?i ph?c model:', BEST_MODEL)
print(f'IMGSZ={IMGSZ_USED}, BATCH={BATCH_USED}, CONF={CONF_BEST}, IOU={IOU_BEST}')

RESULT_ZIP = WORK / 'visdrone_privacy_results.zip'
def create_results_zip():
    roots = [ARTIFACT, RUNS_DIR]
    files = [DATA_YAML, SEQ_INDEX, WORK / 'bytetrack_person.yaml']
    with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for root in roots:
            if root.is_dir():
                for path in root.rglob('*'):
                    if path.is_file() and path != RESULT_ZIP:
                        zf.write(path, path.relative_to(WORK))
        for path in files:
            if path.is_file() and path != RESULT_ZIP:
                zf.write(path, path.relative_to(WORK))
    return RESULT_ZIP


In [ ]:
# CELL 7/13 Â· ByteTrack -> tracks_frame.csv (chunked, auto-merge)
# Input : yolo26s.pt, sequence_index.csv
# Output: artifacts/tracks_frame.csv (merged from tracks_chunk_*.csv)
#         outputs/tracks_chunk_XXX.csv, outputs/chunk_state.json
#
# Chunking mechanism:
#   - Split sequences into small chunks (CHUNK_SIZE sequences/chunk)
#   - Each chunk saved to separate file -> no progress loss on RAM limit
#   - If RAM >= RAM_STOP_GIB: save current chunk, stop safely, save state
#   - Re-run cell to continue from next chunk (resume)
#   - After all chunks: merge all tracks_chunk_*.csv -> tracks_frame.csv

def _read_int(path):
    try:
        return int(Path(path).read_text().strip())
    except Exception:
        return None

def _cgroup_used_gib():
    cur = _read_int('/sys/fs/cgroup/memory.current') or _read_int(
        '/sys/fs/cgroup/memory/memory.usage_in_bytes')
    if cur is None:
        return None
    inactive = 0
    for stat in ('/sys/fs/cgroup/memory.stat', '/sys/fs/cgroup/memory/memory.stat'):
        try:
            for ln in Path(stat).read_text().splitlines():
                key, _, val = ln.partition(' ')
                if key in ('inactive_file', 'total_inactive_file'):
                    inactive = int(val)
                    break
            break
        except Exception:
            continue
    return max(cur - inactive, 0) / 2**30

def _cgroup_limit_gib():
    for path in ('/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory/memory.limit_in_bytes'):
        v = _read_int(path)
        if v is not None and v < 2**60:
            return v / 2**30
    return None

def _mem_used_gib():
    used = _cgroup_used_gib()
    if used is not None:
        return used
    if psutil is not None:
        try:
            return psutil.Process().memory_info().rss / 2**30
        except Exception:
            return None
    return None

def _mem_snapshot():
    snap = {}
    if psutil is not None:
        try:
            snap['rss'] = psutil.Process().memory_info().rss / 2**30
            snap['avail'] = psutil.virtual_memory().available / 2**30
        except Exception:
            pass
    try:
        snap['objs'] = float(len(gc.get_objects()))
    except Exception:
        pass
    used = _mem_used_gib()
    if used is not None:
        snap['used'] = used
    try:
        snap['vram'] = torch.cuda.memory_allocated() / 2**30
    except Exception:
        pass
    return snap

def _vram_free_gib(idx):
    try:
        return torch.cuda.mem_get_info(idx)[0] / 2**30
    except Exception:
        return 0.0

def _trim_host_memory():
    gc.collect()
    if _LIBC is not None:
        try:
            _LIBC.malloc_trim(0)
        except Exception:
            pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _same_device(pred, device):
    try:
        d = pred.device
        if str(device) == 'cpu':
            return d.type == 'cpu'
        return d.type == 'cuda' and int(d.index or 0) == int(device)
    except Exception:
        return False

def _reset_trackers(model, device):
    pred = getattr(model, 'predictor', None)
    if pred is None:
        return
    if not _same_device(pred, device):
        model.predictor = None
        _trim_host_memory()
        return
    if hasattr(pred, 'trackers'):
        del pred.trackers
    gc.collect()

class TrackMemoryStop(Exception):
    """Stop when memory cleanup does not bring RAM below threshold."""

# --- 1. Clean VRAM/RAM from Cell 5-6 ---
CHUNK_SIZE = int(globals().get('TRACK_CHUNK_SIZE', 5))  # sequences per chunk
CHUNK_DIR = Path('/kaggle/working/outputs')
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = CHUNK_DIR / 'chunk_state.json'
CHUNK_PREFIX = 'tracks_chunk_'

# RAM threshold (reuse cell config if available)
CHUNK_RAM_LIMIT_GIB = float(globals().get('TRACK_RAM_LIMIT_GB') or 0.0) or _cgroup_limit_gib()
if not CHUNK_RAM_LIMIT_GIB and psutil is not None:
    try:
        CHUNK_RAM_LIMIT_GIB = psutil.virtual_memory().total / 2**30
    except Exception:
        CHUNK_RAM_LIMIT_GIB = None
CHUNK_RAM_STOP_FRAC = float(globals().get('TRACK_RAM_STOP_FRAC', 0.85))
CHUNK_RAM_STOP_GIB = CHUNK_RAM_LIMIT_GIB * CHUNK_RAM_STOP_FRAC if CHUNK_RAM_LIMIT_GIB else None
CHUNK_RAM_RESUME_GIB = (CHUNK_RAM_LIMIT_GIB * max(CHUNK_RAM_STOP_FRAC - 0.10, 0.50)) if CHUNK_RAM_LIMIT_GIB else None
print('Chunking:', CHUNK_SIZE, 'seq/chunk | RAM stop=', CHUNK_RAM_STOP_GIB, 'G' if CHUNK_RAM_STOP_GIB else 'RAM guard off')

# --- RAM/VRAM helpers ---
for _n in ('eval_model', 'sweep_model', 'track_model', 'model', 'exp',
           'trainer', 'results', 'val_results', 'sweep_results',
           'sweep_rows', 'det_results', 'metrics', 'fig', 'ax', 'predictor'):
    globals().pop(_n, None)
_trim_host_memory()
if torch.cuda.is_available():
    torch.cuda.ipc_collect()
# â”€â”€ Aggressive VRAM clear on ALL GPUs (Cells 5-6 may leave weights on '0,1') â”€â”€
if torch.cuda.is_available():
    for _gi in range(torch.cuda.device_count()):
        try:
            with torch.cuda.device(_gi):
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
        except Exception:
            pass
    print('VRAM cleared:', ' | '.join(
        f'GPU{i}={torch.cuda.mem_get_info(i)[0]/2**30:.1f}GiB free'
        for i in range(torch.cuda.device_count())))

# --- 2. Select least-loaded GPU on T4 x2 ---
TRACK_DEVICE = DEVICE_EVAL
TRACK_HALF = True
if torch.cuda.is_available():
    _cands = sorted(((_vram_free_gib(i), i) for i in range(torch.cuda.device_count())),
                    key=lambda x: x[1])
    print('VRAM free:', ' | '.join(f'GPU{i}={f:.1f}GiB' for f, i in _cands))
    _best_free, _best_idx = max(_cands)
    if _best_free >= 4.0:
        TRACK_DEVICE = _best_idx
    else:
        TRACK_DEVICE, TRACK_HALF = 'cpu', False
else:
    TRACK_DEVICE, TRACK_HALF = 'cpu', False


# --- Tracker config YAML ---
TRACKER_YAML = WORK / 'bytetrack_person.yaml'
TRACKER_YAML.write_text(yaml.safe_dump({
    'tracker_type': 'bytetrack',
    'track_high_thresh': 0.35,
    'track_low_thresh': 0.05,
    'new_track_thresh': 0.25,
    'track_buffer': TRACK_BUFFER,
    'match_thresh': 0.7,
    'fuse_score': True,
}, sort_keys=False), encoding='utf-8')

# --- 3. Resolution: keep train imgsz ---
TRACK_IMGSZ = int(globals().get('IMGSZ_USED', IMGSZ))
# Resolution ladder for tracking: start at train imgsz, fall back on OOM.
# Tracking works fine at 640â€“800; this avoids OOM on a 14 GiB T4.
_ladder = []
for _s in (TRACK_IMGSZ, 960, 800, 640):
    if _s not in _ladder:
        _ladder.append(_s)
TRACK_PLANS = [(s, TRACK_DEVICE, TRACK_HALF) for s in _ladder]
plan_idx = 0
TRACK_IMGSZ, TRACK_DEVICE, TRACK_HALF = TRACK_PLANS[plan_idx]

CONF_TRACK = max(float(globals().get('CONF_BEST', TRACK_CONF)), TRACK_CONF)
IOU_TRACK = float(globals().get('IOU_BEST', TRACK_IOU))
TRACK_RESUME = bool(globals().get('TRACK_RESUME', True))
print('tracking: imgsz=', TRACK_PLANS[0][0], '(train', TRACK_IMGSZ, ') conf=', CONF_TRACK,
      'iou=', IOU_TRACK, 'max_det=', MAX_DET, 'half=', TRACK_HALF, 'resume=', TRACK_RESUME)
print('Fixed imgsz:', TRACK_IMGSZ, '| device:', TRACK_DEVICE, '| half:', TRACK_HALF)

FIELDS = ['split', 'sequence', 'frame', 'track_id', 'class_id', 'confidence',
          'x1', 'y1', 'x2', 'y2', 'width', 'height']

_st = {'model': None, 'cb': {}, 'cb_base': {}, 'resets': 0, 'resets_total': 0}

def _hard_reset():
    """Release model + clean memory. True if RAM below resume threshold."""
    _st['model'] = None
    _st['cb'] = {}
    _st['cb_base'] = {}
    _trim_host_memory()
    if torch.cuda.is_available():
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    used = _mem_used_gib()
    if CHUNK_RAM_RESUME_GIB is None:
        return True
    return used is not None and used < CHUNK_RAM_RESUME_GIB

# RAM check interval (frames) inside _track_sequence
MEM_EVERY = int(globals().get('TRACK_MEM_CHECK_EVERY', 25))

def _track_sequence(model, split, seq, pairs, imgsz, device, half):
    """Track 1 sequence. Return (rows, n_frames)."""

    frame_of = {str(p[1]): p[0] for p in pairs}
    size_of = {str(p[1]): (p[2], p[3]) for p in pairs}
    _reset_trackers(model, device)
    rows = []
    n = 0
    gen = None
    try:
        gen = model.track(
            source=[str(p[1]) for p in pairs], persist=True, tracker=str(TRACKER_YAML),
            conf=CONF_TRACK, iou=IOU_TRACK, imgsz=imgsz, max_det=MAX_DET,
            batch=1, half=half, device=device, stream=True, stream_buffer=False,
            vid_stride=1, verbose=False,
        )
        for res in gen:
            n += 1
            if MEM_EVERY and n % MEM_EVERY == 0 and _over_mem_chunk():
                raise TrackMemoryStop(
                    f'RAM {_mem_used_gib():.1f}G >= threshold {CHUNK_RAM_STOP_GIB:.1f}G '
                    f'at {split}/{seq} frame {n}')
            path = str(getattr(res, 'path', ''))
            frame = frame_of.get(path)
            if frame is None:
                frame = pairs[min(n, len(pairs) - 1)][0]
            iw, ih = size_of.get(path, (0, 0))
            b = res.boxes
            if b is None or len(b) == 0:
                del b, res
                continue
            xy = b.xyxy.cpu().numpy()
            cf = b.conf.cpu().numpy()
            cl = b.cls.int().cpu().numpy()
            ids = b.id.int().cpu().numpy() if b.id is not None else -np.ones(len(xy), int)
            for box, s, c, t in zip(xy, cf, cl, ids):
                rows.append((split, seq, frame, int(t), int(c), round(float(s), 6),
                             round(float(box[0]), 2), round(float(box[1]), 2),
                             round(float(box[2]), 2), round(float(box[3]), 2),
                             iw, ih))
            del b, xy, cf, cl, ids, res
    finally:
        if gen is not None:
            try:
                gen.close()
            except Exception:
                pass
        try:
            model.predictor = None
        except Exception:
            pass
        _trim_host_memory()
    return rows, n

def _over_mem_chunk():
    if CHUNK_RAM_STOP_GIB is None:
        return False
    used = _mem_used_gib()
    return used is not None and used >= CHUNK_RAM_STOP_GIB

def _write_track_config(final):
    TRACK_CONFIG_JSON = ARTIFACT / 'track_config.json'
    TRACK_CONFIG_JSON.write_text(json.dumps({
        'track_imgsz': TRACK_IMGSZ,
        'conf': CONF_TRACK, 'iou': IOU_TRACK, 'max_det': MAX_DET,
        'track_buffer': TRACK_BUFFER,
        'half': bool(TRACK_HALF), 'device': str(TRACK_DEVICE),
        'ladder': [{'imgsz': s, 'device': str(d), 'half': bool(h)} for s, d, h in TRACK_PLANS],
        'ram_limit_gib': CHUNK_RAM_LIMIT_GIB,
        'ram_stop_gib': CHUNK_RAM_STOP_GIB,
        'auto_resets': _st['resets_total'],
        'completed': bool(final),
        'chunk_dir': str(CHUNK_DIR),
    }, indent=2, ensure_ascii=False), encoding='utf-8')
    return TRACK_CONFIG_JSON

# --- Read chunk resume state ---
done_chunks = []
if STATE_FILE.is_file():
    try:
        _stt = json.loads(STATE_FILE.read_text(encoding='utf-8'))
        done_chunks = _stt.get('done_chunks', [])
        print('Resume chunk: already completed', len(done_chunks), 'chunks')
    except Exception as _e:
        print('Chunk state file corrupt, restarting from scratch:', _e)
        done_chunks = []

df_idx = pd.read_csv(SEQ_INDEX, dtype={'split': 'string', 'sequence': 'string',
                                       'frame': 'int32', 'width': 'int32',
                                       'height': 'int32'})
df_idx = (df_idx[df_idx['split'].isin(list(TRACK_SPLITS))]
          .sort_values(['split', 'sequence', 'frame'], kind='stable'))

all_groups = [(str(sp), str(sq), g)
              for (sp, sq), g in df_idx.groupby(['split', 'sequence'], sort=True, observed=True)]

already_done_seqs = set()
for _cf in sorted(CHUNK_DIR.glob(f'{CHUNK_PREFIX}*.csv')):
    try:
        _df_prev = pd.read_csv(_cf, usecols=['split', 'sequence'],
                              dtype={'split': 'string', 'sequence': 'string'})
        for _r in _df_prev.itertuples():
            already_done_seqs.add((str(_r.split), str(_r.sequence)))
    except Exception:
        pass

pending = [g for g in all_groups if (g[0], g[1]) not in already_done_seqs]
print('Total sequences:', len(all_groups), '| already tracked:', len(already_done_seqs),
      '| remaining:', len(pending), '| frames:', sum(len(g) for _s, _q, g in pending))

def _chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

chunks = list(_chunk_list(pending, CHUNK_SIZE))
print('Split into', len(chunks), 'chunks (size', CHUNK_SIZE, 'sequences/chunk)')
MAX_CHUNKS_PER_RUN = int(globals().get('TRACK_MAX_CHUNKS_PER_RUN', 12))
print('Max chunks this run:', MAX_CHUNKS_PER_RUN)

def _save_chunk(chunk_idx, rows):
    chunk_csv = CHUNK_DIR / f'{CHUNK_PREFIX}{chunk_idx:03d}.csv'
    _new = not chunk_csv.is_file() or chunk_csv.stat().st_size == 0
    with open(chunk_csv, 'a', newline='', encoding='utf-8') as _f:
        _w = csv.writer(_f)
        if _new:
            _w.writerow(FIELDS)
        _w.writerows(rows)
        _f.flush()
        os.fsync(_f.fileno())
    print('  â””â”€ Save chunk', chunk_idx, ':', len(rows), 'rows ->', chunk_csv.name)
    return chunk_csv

def _save_state(finished_chunks):
    STATE_FILE.write_text(json.dumps({
        'done_chunks': finished_chunks,
        'total_chunks': len(chunks),
        'chunk_size': CHUNK_SIZE,
    }, indent=2, ensure_ascii=False), encoding='utf-8')

# --- Main loop over chunks ---
used_plan = {}
plan_idx = 0
t0 = time.time()
n_done = 0
total_rows = 0
mem_stop = False
mem_stop_msg = ''
mem0 = _mem_snapshot()
chunks_this_run = 0

for chunk_idx, chunk_groups in enumerate(chunks):
    if chunk_idx in done_chunks:
        continue

    # RAM check before chunk
    if _over_mem_chunk():
        if _st['resets'] < 6 and _hard_reset():
            _st['resets'] += 1
            _st['resets_total'] += 1
            print('  â†» RAM threshold before chunk', chunk_idx, '-> cleaned model, continue')
        else:
            mem_stop = True
            mem_stop_msg = ('RAM', _mem_used_gib(), 'G >= threshold', CHUNK_RAM_STOP_GIB, 'G '
                            'before chunk', chunk_idx)
            break

    chunk_rows = []
    chunk_done = 0

    for split, seq, g in chunk_groups:
        pairs = [(int(r.frame), BUILD_DIR / split / 'images' / r.flat_name,
                  int(r.width), int(r.height)) for r in g.sort_values('frame').itertuples()]
        pairs = [p for p in pairs if p[1].exists()]
        if not pairs:
            print(f'  {split:<5} {seq:<24} skip (no image)')
            continue

        if _st['model'] is None:
            _st['model'] = YOLO(str(BEST_MODEL))
            _st['cb'] = getattr(_st['model'], 'callbacks', {})
            _st['cb_base'] = {k: len(v) for k, v in _st['cb'].items()}

        rows, n_seen, ok, last_err = None, 0, False, None
        stop_here, stop_msg = False, ''
        try:
            rows, n_seen = _track_sequence(_st['model'], split, seq, pairs, TRACK_IMGSZ, TRACK_DEVICE, TRACK_HALF)
            used_plan[f'{split}/{seq}'] = {'imgsz': TRACK_IMGSZ, 'device': str(TRACK_DEVICE), 'half': bool(TRACK_HALF)}
            ok = True
            plan_idx = 0
        except RuntimeError as _re:
            if 'out of memory' in str(_re).lower() and plan_idx < len(TRACK_PLANS) - 1:
                plan_idx += 1
                TRACK_IMGSZ, TRACK_DEVICE, TRACK_HALF = TRACK_PLANS[plan_idx]
                _st['model'] = None
                _trim_host_memory()
                print(f'  â†» CUDA OOM at imgsz={TRACK_PLANS[plan_idx-1][0]}, retrying {split}/{seq} '
                      f'with imgsz={TRACK_IMGSZ} device={TRACK_DEVICE}')
                del rows, pairs, g
                continue
            raise
        except TrackMemoryStop as _ms:
            stop_here, stop_msg = True, str(_ms)

        if stop_here:
            if _st['resets'] < 6 and _hard_reset():
                _st['resets'] += 1
                _st['resets_total'] += 1
                print('  â†»', stop_msg, '-> cleaned model, retry this sequence')
                del rows, pairs, g
                continue
            mem_stop = True
            mem_stop_msg = stop_msg
            print('  ! Cannot lower RAM after cleanup -> stop at', split, '/', seq,
                  '(sequence NOT saved, will track next run)')
            del rows, pairs, g
            _trim_host_memory()
            break

        if not ok:
            raise RuntimeError(f'OOM at fixed imgsz={TRACK_IMGSZ} for {split}/{seq}: {last_err}')

        chunk_rows.extend(rows)
        chunk_done += 1
        n_done += 1
        total_rows += len(rows)
        _st['resets'] = 0

        # Trim stale callbacks
        for _k, _n_cb in _st['cb_base'].items():
            _lst = _st['cb'].get(_k)
            if _lst is not None and len(_lst) > _n_cb:
                del _lst[_n_cb:]

        del rows, pairs, g
        _trim_host_memory()

        _plan = used_plan[f'{split}/{seq}']
        _mem = _mem_snapshot()
        _used = _mem.get('used', float('nan'))
        _pct = f'{_used / CHUNK_RAM_LIMIT_GIB:.0%}' if (CHUNK_RAM_LIMIT_GIB and _used == _used) else '--'
        _eta = (time.time() - t0) / n_done * (len(pending) - n_done) / 60.0 if n_done else 0
        print(f'  {split:<5} {seq:<24} {n_seen:>4} frame | {total_rows:>7} box | '
              f'imgsz={_plan["imgsz"]:<4} | '
              f'RAM {_used:5.2f}G/{CHUNK_RAM_LIMIT_GIB or 0:4.1f}G ({_pct}) Î”{_used - mem0.get("used", 0):+5.2f} | '
              f'RSS {_mem.get("rss", float("nan")):5.2f}G | '
              f'objs {_mem.get("objs", 0) / 1000:6.0f}k | '
              f'VRAM {_mem.get("vram", float("nan")):4.2f}G | ETA {_eta:5.1f} min')

    # Save this chunk to file
    if chunk_rows:
        _save_chunk(chunk_idx, chunk_rows)

    # Mark chunk as done
    done_chunks.append(chunk_idx)
    _save_state(done_chunks)

    # Release memory after each chunk
    _st['model'] = None
    _trim_host_memory()

    chunks_this_run += 1
    if chunks_this_run >= MAX_CHUNKS_PER_RUN:
        print(f'  -> Da xu ly {chunks_this_run} chunks lan chay nay, dung lai de merge')
        break
    if mem_stop:
        break

# --- End: safe stop or merge ---
if mem_stop:
    _left = len(pending) - n_done
    print('\n' + '=' * 78)
    print('SAFE STOP:', mem_stop_msg)
    print('Saved', n_done, 'sequences /', total_rows, 'rows into chunk files.')
    print('Remaining', _left, 'sequences not tracked. Re-run this cell to resume.')
    print('To free RAM completely: Restart session -> re-run Cell 7.')
    print('=' * 78)
    print('track_config.json ->', _write_track_config(False))
    _trim_host_memory()
    raise TrackMemoryStop(
        f'RAM still above threshold after {_st["resets_total"]} cleanups. '
        f'Saved {n_done} sequences into chunks -> re-run this cell to continue '
        f'{_left} remaining sequences.')

# --- Merge all chunks into tracks_frame.csv ---
chunk_files = sorted(CHUNK_DIR.glob(f'{CHUNK_PREFIX}*.csv'))
if not chunk_files:
    raise RuntimeError('No chunk files to merge. Possibly no chunk ran yet.')

print('\nMerging', len(chunk_files), 'chunk files ->', TRACKS_CSV.name, '...')
_DT = {'split': 'string', 'sequence': 'string', 'frame': 'int32', 'track_id': 'int32',
       'class_id': 'int8', 'confidence': 'float32', 'x1': 'float32', 'y1': 'float32',
       'x2': 'float32', 'y2': 'float32', 'width': 'int32', 'height': 'int32'}

_all_dfs = []
for _cf in chunk_files:
    _df = pd.read_csv(_cf, dtype=_DT)
    _all_dfs.append(_df)

df_all = pd.concat(_all_dfs, ignore_index=True)
df_all = df_all.sort_values(['split', 'sequence', 'frame', 'track_id'], kind='stable')
df_all.to_csv(TRACKS_CSV, index=False)

print('tracks_frame.csv:', len(df_all), 'rows |', df_all.track_id.nunique(), 'track_id')
print(df_all.groupby('split').agg(frames=('frame', 'nunique'),
                                  rows=('track_id', 'size')).to_string())

# Keep chunk files for audit (do not delete)

print('track_config.json ->', _write_track_config(True))

_trim_host_memory()


In [ ]:
# ── CELL 8/13 · Trích đặc trưng mỗi frame (causal, khớp runtime) ──
# Vào : tracks_frame.csv, sequence_index.csv, gt_boxes.csv
# Ra  : features_frame.csv, feature_schema.json
#
# Bộ đặc trưng schema v2: 25 đặc trưng chính + 4 đặc trưng chẩn đoán.
#   - Giữ causal tuyệt đối: mỗi dòng chỉ dùng frame <= t.
#   - 4 đặc trưng bị loại khỏi model (track_state, aspect_dev, is_trunc,
#     frame_det_delta) vẫn được tính để cell metrics so sánh univariate AUC.
#   - Các đặc trưng mới đều tính được ở runtime từ TrackHistory:
#     conf_mean5, conf_slope5, area_norm, height_norm, center_y_norm,
#     iou_prev, scale_change, crop_contrast.
#   - 1 detection -> nearest/n_neighbors/overlap = 0 (khớp step5 runtime).
#   - Lịch sử mỗi track giữ tối đa HIST_KEEP phần tử.


def crop_gray(img, box, resize=CROP_RESIZE):
    if img is None:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    x1 = int(max(0, min(math.floor(x1), w - 1)))
    y1 = int(max(0, min(math.floor(y1), h - 1)))
    x2 = int(max(x1 + 1, min(math.ceil(x2), w)))
    y2 = int(max(y1 + 1, min(math.ceil(y2), h)))
    if x2 <= x1 or y2 <= y1:
        return None
    c = img[y1:y2, x1:x2]
    if resize and max(c.shape[:2]) > resize:
        s = resize / max(c.shape[:2])
        c = cv2.resize(c, (max(1, int(c.shape[1] * s)), max(1, int(c.shape[0] * s))),
                       interpolation=cv2.INTER_AREA)
    return c


def lap_var(c):
    if c is None or c.size == 0:
        return 0.0
    return float(cv2.Laplacian(c, cv2.CV_64F).var())


def box_iou(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = max(1e-9, (a[2] - a[0]) * (a[3] - a[1]) +
                (b[2] - b[0]) * (b[3] - b[1]) - inter)
    return float(inter / union)


def conf_slope(values):
    if len(values) < 2:
        return 0.0
    x = np.arange(len(values), dtype=np.float64)
    xm = float(x.mean())
    denom = float(((x - xm) ** 2).sum())
    if denom <= 0:
        return 0.0
    y = np.asarray(values, dtype=np.float64)
    return float(((x - xm) * (y - y.mean())).sum() / denom)


_TRK_SRC = TRACKS_CSV
if not _TRK_SRC.is_file():
    _TRK_ALT = TRACKS_CSV.with_name(TRACKS_CSV.stem + '.partial.csv')
    if not _TRK_ALT.is_file():
        raise FileNotFoundError(
            f'Không thấy {TRACKS_CSV.name} lẫn {_TRK_ALT.name}. Chạy Cell 7/13 trước.')
    _TRK_SRC = _TRK_ALT
    print('!' * 78)
    print(f'CHƯA TRACK XONG: dùng {_TRK_ALT.name} (dữ liệu MỘT PHẦN).')
    print('Chạy lại Cell 7/13 để hoàn tất, rồi chạy lại các cell sau.')
    print('!' * 78)
df_trk = pd.read_csv(_TRK_SRC)
df_idx = add_validation_split(pd.read_csv(SEQ_INDEX), seed=SEED, validation_fraction=0.20)
df_idx.to_csv(SEQ_INDEX, index=False)
if GT_CSV.is_file():
    _gt = pd.read_csv(GT_CSV)
    _seq_split = df_idx[['sequence', 'split']].drop_duplicates()
    _gt = _gt.drop(columns=['split'], errors='ignore').merge(_seq_split, on='sequence', how='inner')
    _gt.to_csv(GT_CSV, index=False)
df_gt = pd.read_csv(GT_CSV)

meta = {(r.split, r.sequence, int(r.frame)): (int(r.width), int(r.height), r.flat_name)
        for r in df_idx.itertuples()}

train_gt = df_gt[df_gt.split == 'train']
r_ref = float(np.median(train_gt.w / train_gt.h)) if len(train_gt) else 0.4
print('r_ref (median w/h train, chỉ dùng cho đặc trưng chẩn đoán aspect_dev):', round(r_ref, 4))

HIST_KEEP = 64   # số phần tử lịch sử tối đa giữ lại cho mỗi track
DIAGNOSTIC_FEATURES = list(DROPPED_FEATURES)

rows = []
for (split, seq), g in df_trk.groupby(['split', 'sequence'], sort=True):
    g = g.sort_values(['frame', 'track_id'])
    state = {}
    prev_n = None
    ewma = None

    for frame, fsub in g.groupby('frame', sort=True):
        frame = int(frame)
        iw, ih, flat = meta.get((split, seq, frame), (0, 0, None))
        if iw == 0 or flat is None:
            continue
        D = math.hypot(iw, ih)
        img = cv2.imread(str(BUILD_DIR / split / 'images' / flat), cv2.IMREAD_GRAYSCALE)

        boxes = fsub[['x1', 'y1', 'x2', 'y2']].to_numpy(float)
        N = len(boxes)
        cx = (boxes[:, 0] + boxes[:, 2]) / 2.0
        cy = (boxes[:, 1] + boxes[:, 3]) / 2.0
        centers = np.stack([cx, cy], 1)
        areas = (np.clip(boxes[:, 2] - boxes[:, 0], 1e-6, None) *
                 np.clip(boxes[:, 3] - boxes[:, 1], 1e-6, None))

        dx = centers[:, None, 0] - centers[None, :, 0]
        dy = centers[:, None, 1] - centers[None, :, 1]
        dist = np.hypot(dx, dy)
        np.fill_diagonal(dist, np.inf)
        nearest = np.clip(dist.min(1) / D, 0, 1) if N > 1 else np.zeros(N)
        radius = NEIGHBOR_RADIUS_FRAC * D
        n_nb = np.clip((dist <= radius).sum(1) / NEIGHBOR_K, 0, 1) if N > 1 else np.zeros(N)

        x1 = np.maximum(boxes[:, None, 0], boxes[None, :, 0])
        y1 = np.maximum(boxes[:, None, 1], boxes[None, :, 1])
        x2 = np.minimum(boxes[:, None, 2], boxes[None, :, 2])
        y2 = np.minimum(boxes[:, None, 3], boxes[None, :, 3])
        inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
        union = areas[:, None] + areas[None, :] - inter
        iou_mat = np.where(union > 0, inter / union, 0.0)
        np.fill_diagonal(iou_mat, 0.0)
        ov_max = iou_mat.max(1) if N > 1 else np.zeros(N)

        delta = 0.0 if prev_n is None else (N - prev_n) / (prev_n + 1)
        expected = N if ewma is None else ewma
        ratio = N / (expected + 1.0)
        ewma = N if ewma is None else EWMA_ALPHA * N + (1 - EWMA_ALPHA) * ewma
        prev_n = N

        for j, r in enumerate(fsub.itertuples()):
            key = (split, seq, int(r.track_id))
            st = state.get(key)
            if st is None:
                st = {'first': frame, 'prev': None, 'n': 0, 'conf': [], 'cen': [],
                      'lap': None, 'prev_box': None, 'prev_area': None}
                state[key] = st

            st['n'] += 1
            age = frame - st['first'] + 1
            tsu = 0 if st['prev'] is None else max(0, frame - st['prev'] - 1)
            hit = min(1.0, st['n'] / max(age, 1))

            conf = float(r.confidence)
            c5 = st['conf'][-(CONF_WINDOW - 1):] + [conf] if CONF_WINDOW > 1 else [conf]
            conf_std5 = float(np.std(c5)) if len(c5) > 1 else 0.0
            conf_delta = 0.0 if not st['conf'] else conf - st['conf'][-1]
            conf_mean5 = float(np.mean(c5)) if c5 else conf
            conf_slope5 = conf_slope(c5)

            box = (boxes[j, 0], boxes[j, 1], boxes[j, 2], boxes[j, 3])
            bw = max(box[2] - box[0], 1e-6)
            bh = max(box[3] - box[1], 1e-6)
            area = bw * bh
            min_side = min(bw, bh)
            aspect = abs(bw / bh - r_ref)
            area_norm = area / max(iw * ih, 1.0)
            height_norm = bh / max(ih, 1.0)
            center_y_norm = box[1] + bh / 2.0
            center_y_norm = float(center_y_norm / max(ih, 1.0))
            de = float(max(0.0, min(box[0], box[1], iw - box[2], ih - box[3]) / D))
            trunc = 1.0 if min(box[0], box[1], iw - box[2], ih - box[3]) <= TRUNC_MARGIN_PX else 0.0

            speed = 0.0
            if st['prev'] is not None:
                speed = float(np.hypot(cx[j] - st['cen'][-1][0], cy[j] - st['cen'][-1][1]) / D)

            cen_win = st['cen'][-(PRED_UNCERT_WINDOW - 1):] + [(cx[j], cy[j])] if PRED_UNCERT_WINDOW > 1 else [(cx[j], cy[j])]
            unc = 0.0
            if len(cen_win) > 1:
                arr = np.array(cen_win)
                unc = float(np.sqrt(arr[:, 0].var() + arr[:, 1].var()) / D)

            # Độ mượt track + thay đổi scale so với frame trước
            iou_prev = box_iou(st['prev_box'], box) if st['prev_box'] is not None else 0.0
            if st['prev_area'] is not None and st['prev_area'] > 0:
                scale_change = float(np.clip(math.log(max(area, 1e-6) / st['prev_area']), -5.0, 5.0))
            else:
                scale_change = 0.0

            crop = crop_gray(img, box)
            lv = lap_var(crop)
            lap_log = float(math.log1p(lv))
            bright = float(crop.mean() / 255.0) if crop is not None and crop.size else 0.0
            contrast = float(crop.std() / 255.0) if crop is not None and crop.size else 0.0
            blur = 0.0
            if st['lap'] is not None and (st['lap'] + lv) > 0:
                blur = float((st['lap'] - lv) / (st['lap'] + lv + 1e-6))

            tstate = 0.0 if age == 1 else (1.0 if tsu == 0 else 2.0)

            rows.append({
                'split': split, 'sequence': seq, 'frame': frame, 'track_id': int(r.track_id),
                # --- 25 đặc trưng model ---
                'conf': conf, 'conf_std5': conf_std5, 'conf_delta': conf_delta,
                'conf_mean5': conf_mean5, 'conf_slope5': conf_slope5,
                'lap_var_log': lap_log, 'brightness': bright,
                'motion_blur_ratio': blur, 'crop_contrast': contrast,
                'track_age': float(age), 'hit_ratio': float(hit),
                'time_since_update': float(tsu),
                'speed_norm': speed, 'pred_uncert': unc,
                'iou_prev': float(iou_prev), 'scale_change': scale_change,
                'min_side_px': float(min_side), 'area_norm': float(area_norm),
                'height_norm': float(height_norm), 'center_y_norm': float(center_y_norm),
                'dist_edge': de,
                'nearest_dist_norm': float(nearest[j]),
                'n_neighbors_r2_norm': float(n_nb[j]),
                'overlap_max': float(ov_max[j]),
                'n_det_expected_ratio': float(ratio),
                # --- 4 đặc trưng chẩn đoán (bị loại khỏi model) ---
                'track_state': tstate, 'aspect_dev': float(aspect),
                'is_trunc': trunc, 'frame_det_delta': float(delta),
            })

            st['prev'] = frame
            st['conf'].append(conf)
            st['cen'].append((float(cx[j]), float(cy[j])))
            st['lap'] = lv
            st['prev_box'] = box
            st['prev_area'] = area
            if len(st['conf']) > HIST_KEEP:
                st['conf'] = st['conf'][-HIST_KEEP:]
                st['cen'] = st['cen'][-HIST_KEEP:]

ALL_FEATURE_COLS = list(FEATURES) + DIAGNOSTIC_FEATURES
df_feat = repair_tracker_features(pd.DataFrame(rows))
df_feat[ALL_FEATURE_COLS] = (df_feat[ALL_FEATURE_COLS]
                             .replace([np.inf, -np.inf], np.nan)
                             .fillna(0.0)
                             .astype(np.float32))
df_feat.to_csv(FEATURES_CSV, index=False)
print('features_frame.csv:', len(df_feat), 'dòng |',
      len(FEATURES), 'đặc trưng model +', len(DIAGNOSTIC_FEATURES), 'chẩn đoán')

schema = {
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'features': FEATURES,
    'dropped_features': DROPPED_FEATURES,
    'drop_rationale': FEATURE_DROP_RATIONALE,
    'feature_groups': FEATURE_GROUPS,
    'tracks_source': _TRK_SRC.name,
    'tracks_complete': bool(_TRK_SRC == TRACKS_CSV),
    'count': len(FEATURES),
    'window': {'size': WINDOW_SIZE, 'stride': WINDOW_STRIDE, 'min_frames': MIN_WINDOW_FRAMES},
    'sample_aggregation': FEATURE_AGG,
    'causality': 'mỗi dòng chỉ dùng frame <= t; sample tại cutoff chỉ dùng frame <= cutoff',
    'aggregation': {f: FEATURE_AGG for f in FEATURES},
    'definitions': {
        'conf': 'confidence của detection tại frame t',
        'conf_std5': 'std của confidence trong CONF_WINDOW frame quá khứ',
        'conf_delta': 'conf_t - conf_{t-1}',
        'conf_mean5': 'trung bình confidence trong CONF_WINDOW frame gần nhất',
        'conf_slope5': 'độ dốc hồi quy tuyến tính của confidence trên CONF_WINDOW frame',
        'track_age': 't - frame đầu tiên của track + 1',
        'hit_ratio': 'số frame track xuất hiện / track_age',
        'time_since_update': 'số frame trống kể từ lần xuất hiện trước',
        'min_side_px': 'min(w, h) theo pixel',
        'area_norm': 'diện tích box / diện tích ảnh',
        'height_norm': 'chiều cao box / chiều cao ảnh',
        'center_y_norm': 'tâm box theo trục dọc / chiều cao ảnh',
        'dist_edge': 'khoảng cách chuẩn hoá tới biên ảnh gần nhất',
        'speed_norm': 'dịch chuyển tâm box / đường chéo ảnh',
        'pred_uncert': 'PROXY: std(center) trên PRED_UNCERT_WINDOW frame / D',
        'iou_prev': 'IoU với box frame trước (0 nếu là frame đầu)',
        'scale_change': 'log(area_t / area_{t-1}), clip [-5, 5]',
        'lap_var_log': 'log1p(Var(Laplacian(crop gray)))',
        'brightness': 'mean(crop gray) / 255',
        'crop_contrast': 'std(crop gray) / 255',
        'motion_blur_ratio': '(L_{t-1} - L_t) / (L_{t-1} + L_t + eps)',
        'nearest_dist_norm': 'khoảng cách tới tâm gần nhất / D (0 nếu chỉ có 1 detection)',
        'n_neighbors_r2_norm': 'min(số tâm trong bán kính r / K, 1)',
        'overlap_max': 'IoU lớn nhất với box khác trong frame',
        'n_det_expected_ratio': 'N_t / (EWMA(N) + 1), EWMA chỉ dùng quá khứ',
        # chẩn đoán
        'track_state': '0 tentative, 1 tracked, 2 lost (tái dựng từ output ByteTrack)',
        'aspect_dev': 'abs(w/h - r_ref), r_ref = median w/h GT train',
        'is_trunc': '1 nếu box chạm biên trong TRUNC_MARGIN_PX',
        'frame_det_delta': '(N_t - N_{t-1}) / (N_{t-1} + 1)',
    },
    'notes': [
        'pred_uncert là proxy, không phải trace covariance Kalman.',
        '4 đặc trưng chẩn đoán vẫn được tính để cell metrics so sánh univariate AUC, nhưng không đưa vào XGBoost.',
        'nearest/overlap/n_neighbors = 0 khi frame chỉ có 1 detection (khớp step5).',
        'CROP_RESIZE và NEIGHBOR_RADIUS_FRAC/NEIGHBOR_K được đồng bộ với step5 runtime.',
        'Sample-level dùng FEATURE_AGG (mặc định last) để khớp 1:1 với runtime.',
    ],
}
SCHEMA_JSON.write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding='utf-8')
print('feature_schema.json ->', SCHEMA_JSON)
print(df_feat[list(FEATURES)].describe().T[['mean', 'std', 'min', 'max']].round(4).to_string())


In [ ]:
# ── CELL 9/13 · Ẩn danh + tấn công Re-ID độc lập causal ──
# Vào : tracks_frame.csv, gt_boxes.csv, sequence_index.csv
# Ra  : samples.csv, attack_results.csv, track_identity.csv
#
# Sửa causal: gallery chỉ dùng embedding có frame <= cutoff của mẫu;
# không dùng mean_all toàn bộ track, không dùng far_mean nhìn ngoài cửa sổ.

import torch
import torch.nn.functional as F
from torchvision import transforms


def _iou_matrix(a, b):
    """IoU ma trận giữa các box a (N,4) và b (M,4), vector hóa NumPy."""
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float64)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = np.maximum(1e-9, area_a[:, None] + area_b[None, :] - inter)
    return inter / union


def track_identity(df_trk, df_gt, iou_th):
    """Ánh xạ mỗi track sang identity thật (dựa trên IoU trung bình với GT cùng frame)."""
    rows = []
    gt_by_seq_frame = {}
    for (split, seq, frame), g in df_gt.groupby(['split', 'sequence', 'frame']):
        gt_by_seq_frame[(split, seq, int(frame))] = g
    gt_by_seq = {}
    for (split, seq), g in df_gt.groupby(['split', 'sequence']):
        gt_by_seq[(split, seq)] = g

    for (split, seq, tid), g in df_trk.groupby(['split', 'sequence', 'track_id']):
        g = g.sort_values('frame')
        track_boxes = g[['x1', 'y1', 'x2', 'y2']].to_numpy(float)
        track_frames = g['frame'].to_numpy(int)
        iou_by_gt_id = defaultdict(list)
        for k in range(len(g)):
            f = int(track_frames[k])
            gt_frame = gt_by_seq_frame.get((split, seq, f))
            if gt_frame is None or len(gt_frame) == 0:
                continue
            tb = track_boxes[k:k + 1]
            gb = gt_frame[['x1', 'y1', 'x2', 'y2']].to_numpy(float)
            iou_vec = _iou_matrix(tb, gb)[0]
            for gt_iou, gt_id in zip(iou_vec, gt_frame['target_id'].to_numpy(int)):
                iou_by_gt_id[int(gt_id)].append(float(gt_iou))
        # Fallback: neu khong co GT cung frame nao, dung toan bo GT cua sequence
        if len(iou_by_gt_id) == 0:
            gt_seq = gt_by_seq.get((split, seq))
            if gt_seq is not None and len(gt_seq) > 0:
                gb = gt_seq[['x1', 'y1', 'x2', 'y2']].to_numpy(float)
                gt_ids = gt_seq['target_id'].to_numpy(int)
                iou_mat = _iou_matrix(track_boxes, gb)
                mean_iou = iou_mat.mean(axis=0)
                best_idx = int(np.argmax(mean_iou))
                best_id = int(gt_ids[best_idx])
                best_iou = float(mean_iou[best_idx])
                rows.append(((split, seq, int(tid)), (best_id, best_iou)))
                continue
        if len(iou_by_gt_id) > 0:
            best_id, best_iou = None, 0.0
            for gt_id, ious in iou_by_gt_id.items():
                mi = float(np.mean(ious))
                if mi > best_iou:
                    best_iou, best_id = mi, gt_id
            rows.append(((split, seq, int(tid)), (best_id, best_iou)))
        else:
            rows.append(((split, seq, int(tid)), (None, 0.0)))
    return dict(rows)


def crop_color(img, box, pad=0.05):
    if img is None:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = max(x2 - x1, 1e-6), max(y2 - y1, 1e-6)
    x1 = int(max(0, math.floor(x1 - pad * bw)))
    y1 = int(max(0, math.floor(y1 - pad * bh)))
    x2 = int(min(w, math.ceil(x2 + pad * bw)))
    y2 = int(min(h, math.ceil(y2 + pad * bh)))
    if x2 <= x1 or y2 <= y1:
        return None
    return img[y1:y2, x1:x2]


def anonymize(img):
    """Ap dung phuong phap an danh cau hinh (blur hoac pixelate) vao mot crop."""
    if img is None or img.size == 0:
        return img
    if ANON_METHOD == 'pixelate':
        h, w = img.shape[:2]
        block = max(1, int(ANON_PIXEL_BLOCK))
        # Thu nho roi phong lai de tao hieu ung pixelate
        small = cv2.resize(img, (max(1, w // block), max(1, h // block)),
                           interpolation=cv2.INTER_LINEAR)
        out = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
        return out
    else:  # 'blur'
        k = max(1, int(ANON_KSIZE) | 1)  # dam bao la so le
        return cv2.GaussianBlur(img, (k, k), 0)


@torch.no_grad()
def embed_list(model, tf, device, crops):
    if not crops:
        return np.zeros((0, 1), np.float32)
    outs = []
    for i in range(0, len(crops), REID_BATCH):
        batch = torch.stack([tf(c) for c in crops[i:i + REID_BATCH]]).to(device)
        f = F.normalize(model(batch).float(), dim=1)
        outs.append(f.cpu().numpy())
    return np.concatenate(outs, 0)


def build_embedding_cache(df_trk, model, tf, device):
    cache = defaultdict(lambda: {'frames': [], 'clean': [], 'anon': []})
    df_emb = (df_trk.sort_values(['split', 'sequence', 'track_id', 'frame'])
              .assign(_r=lambda d: d.groupby(['split', 'sequence', 'track_id']).cumcount()))
    df_emb = df_emb[df_emb._r % max(1, EMBED_STRIDE) == 0]
    print('crop cần embed:', len(df_emb), '/', len(df_trk))

    for (split, seq, frame), sub in df_emb.groupby(['split', 'sequence', 'frame'], sort=True):
        img = cv2.imread(str(BUILD_DIR / split / 'images' /
                             f'{seq}__{int(frame):07d}.jpg'), cv2.IMREAD_COLOR)
        if img is None:
            continue
        keys, clean, anon = [], [], []
        for r in sub.itertuples():
            c = crop_color(img, (r.x1, r.y1, r.x2, r.y2))
            if c is None or c.size == 0:
                continue
            keys.append((split, seq, int(r.track_id), int(frame)))
            clean.append(c)
            anon.append(anonymize(c))
        if not keys:
            continue
        ec = embed_list(model, tf, device, clean)
        ea = embed_list(model, tf, device, anon)
        for k, a, b in zip(keys, ec, ea):
            d = cache[(k[0], k[1], k[2])]
            d['frames'].append(k[3])
            d['clean'].append(a)
            d['anon'].append(b)

    for k in list(cache):
        d = cache[k]
        order = np.argsort(d['frames'])
        d['frames'] = np.asarray(d['frames'], int)[order]
        d['clean'] = np.stack(d['clean'])[order].astype(np.float32)
        d['anon'] = np.stack(d['anon'])[order].astype(np.float32)
    return dict(cache)


def build_identity_index(cache, ident):
    per = defaultdict(lambda: defaultdict(lambda: {'frames': [], 'embs': []}))
    for (split, seq, tid), d in cache.items():
        info = ident.get((split, seq, tid))
        if info is None or info[1] < LABEL_MIN_MATCH_RATE:
            continue
        slot = per[split][(seq, info[0])]
        slot['frames'].append(d['frames'])
        slot['embs'].append(d['clean'])

    out = {}
    for split, ids in per.items():
        out[split] = {}
        for key, v in ids.items():
            fr = np.concatenate(v['frames'])
            em = np.concatenate(v['embs'])
            order = np.argsort(fr)
            fr, em = fr[order], em[order]
            # Giới hạn số embedding/identity: track dài không được lấn át gallery
            if GALLERY_MAX_PER_ID and len(fr) > GALLERY_MAX_PER_ID:
                pick = np.linspace(0, len(fr) - 1, GALLERY_MAX_PER_ID).round().astype(int)
                fr, em = fr[pick], em[pick]
            out[split][key] = {'frames': fr, 'cum': np.cumsum(em, 0), 'n': len(fr)}
    return out


def _unit(v):
    n = float(np.linalg.norm(v))
    return v / n if n > 0 else None


def mean_upto(entry, max_frame):
    """Trung bình embedding của các frame <= max_frame (causal, không nhìn tương lai)."""
    fr = entry['frames']
    hi = int(np.searchsorted(fr, max_frame, side='right'))
    if hi == 0:
        return None
    cum = entry['cum']
    total = cum[hi - 1]
    return _unit(total / hi)


# ── Đọc tracks: Cell 7 có thể đã dừng an toàn -> dùng *.partial.csv ──
_TRK_SRC = TRACKS_CSV
if not _TRK_SRC.is_file():
    _TRK_ALT = TRACKS_CSV.with_name(TRACKS_CSV.stem + '.partial.csv')
    if not _TRK_ALT.is_file():
        raise FileNotFoundError(
            f'Không thấy {TRACKS_CSV.name} lẫn {_TRK_ALT.name}. '
            f'Chạy Cell 7/13 trước.')
    _TRK_SRC = _TRK_ALT
    print('!' * 78)
    print(f'CHƯA TRACK XONG: dùng {_TRK_ALT.name} (dữ liệu MỘT PHẦN).')
    print('Chạy lại Cell 7/13 để hoàn tất, rồi chạy lại các cell sau.')
    print('!' * 78)
df_trk = pd.read_csv(_TRK_SRC)
df_gt = pd.read_csv(GT_CSV)

IDENT = track_identity(df_trk, df_gt, TRACK_ID_IOU)
df_tid = pd.DataFrame([{'split': k[0], 'sequence': k[1], 'track_id': k[2],
                        'identity': v[0], 'match_rate': v[1]} for k, v in IDENT.items()])
df_tid.to_csv(ARTIFACT / 'track_identity.csv', index=False)
n_ok = int(((df_tid['identity'].notna()) & (df_tid['match_rate'] >= LABEL_MIN_MATCH_RATE)).sum())
print(f'track_identity: {len(df_tid)} tracks | co identity hop le: {n_ok} | ty le: {n_ok/max(len(df_tid),1):.2%}')
if len(df_tid):
    _mr = df_tid['match_rate'].dropna()
    if len(_mr):
        print(f'  match_rate distribution: min={_mr.min():.3f} | p25={_mr.quantile(0.25):.3f} | median={_mr.median():.3f} | p75={_mr.quantile(0.75):.3f} | max={_mr.max():.3f}')

def build_samples(df_trk, ident):
    """Tạo mẫu sliding-window từ tracks, gán identity từ IDENT."""
    rows = []
    sample_id = 0
    for (split, seq, tid), g in df_trk.groupby(['split', 'sequence', 'track_id']):
        g = g.sort_values('frame')
        frames = g['frame'].to_numpy()
        n = len(frames)
        if n < MIN_WINDOW_FRAMES:
            continue
        info = ident.get((split, seq, int(tid)))
        identity = info[0] if info else np.nan
        match_rate = float(info[1]) if info else 0.0
        start = 0
        while start + WINDOW_SIZE <= n:
            win = frames[start:start + WINDOW_SIZE]
            rows.append({
                'sample_id': sample_id,
                'split': split,
                'sequence': seq,
                'track_id': int(tid),
                'window_start': int(win[0]),
                'window_end': int(win[-1]),
                'identity': identity,
                'identity_match_rate': match_rate,
            })
            sample_id += 1
            start += WINDOW_STRIDE
        if start < n and n - start >= MIN_WINDOW_FRAMES:
            win = frames[start:]
            rows.append({
                'sample_id': sample_id,
                'split': split,
                'sequence': seq,
                'track_id': int(tid),
                'window_start': int(win[0]),
                'window_end': int(win[-1]),
                'identity': identity,
                'identity_match_rate': match_rate,
            })
            sample_id += 1
    df = pd.DataFrame(rows)
    if df.empty:
        df = pd.DataFrame(columns=['sample_id', 'split', 'sequence', 'track_id',
                                   'window_start', 'window_end', 'identity', 'identity_match_rate'])
    return df


df_samples = build_samples(df_trk, IDENT)
df_samples.to_csv(SAMPLES_CSV, index=False)
print('samples:', len(df_samples), '| có identity:',
      int(df_samples.identity.notna().sum()))


def build_reid():
    """Khai báo mô hình Re-ID và BUỘC dùng CUDA nếu có sẵn (T4)."""
    import torchreid
    from torchvision.models import resnet50, ResNet50_Weights

    global DEVICE_EVAL
    # Ép dùng GPU nếu CUDA có sẵn
    if torch.cuda.is_available():
        try:
            torch.cuda.set_device(int(DEVICE_EVAL))
        except Exception:
            pass
        device = f'cuda:{int(DEVICE_EVAL)}'
        try:
            _gname = torch.cuda.get_device_name(int(DEVICE_EVAL))
        except Exception:
            _gname = 'cuda'
        print(f'Re-ID su dung CUDA: {device} | {_gname}')
    else:
        device = 'cpu'
        print('CANH BAO: CUDA khong the, Re-ID se chay tren CPU (cham).')

    model, name = None, None
    if REID_BACKBONE in ('auto', 'osnet_x1_0'):
        try:
            model = torchreid.models.build_model(name='osnet_x1_0', num_classes=1000, pretrained=True)
            model.classifier = torch.nn.Identity()
            name = 'osnet_x1_0'
            print('Re-ID backbone: osnet_x1_0 | device:', device)
        except Exception as e:
            print('OSNet không khả dụng:', str(e)[:140])
            model = None
    if model is None:
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        model.fc = torch.nn.Identity()
        name = 'resnet50'
        print('Re-ID backbone: resnet50 | device:', device)

    model.eval().to(device)
    tf = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((REID_INPUT, REID_INPUT)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return model, tf, device, name

model, tf, device, reid_name = build_reid()
cache = build_embedding_cache(df_trk, model, tf, device)
print('crop đã embed:', len(cache), 'track')
id_index = build_identity_index(cache, IDENT)
print('identity index:', {k: len(v) for k, v in id_index.items()})

rng = np.random.RandomState(SEED)
attack = []
for r in df_samples.itertuples():
    rec = {'sample_id': r.sample_id, 'split': r.split, 'sequence': r.sequence,
           'track_id': int(r.track_id), 'window_start': int(r.window_start),
           'window_end': int(r.window_end), 'identity_true': r.identity,
           'n_probe': 0, 'n_gallery_ids': 0, 'top1_identity': np.nan,
           'top1_sim': np.nan, 'true_sim': np.nan, 'rank_true': np.nan,
           'top1_correct': np.nan, 'probe_consistency': np.nan, 'status': 'ok'}

    if pd.isna(r.identity) or r.identity_match_rate < LABEL_MIN_MATCH_RATE:
        rec['status'] = 'no_identity'
        attack.append(rec)
        continue

    d = cache.get((r.split, r.sequence, int(r.track_id)))
    if d is None:
        rec['status'] = 'no_probe'
        attack.append(rec)
        continue

    # Probe chỉ dùng frame <= cutoff (window_start), không nhìn tương lai.
    cutoff = int(r.window_start)
    sel = np.where(d['frames'] <= cutoff)[0]
    if len(sel) == 0:
        rec['status'] = 'no_probe'
        attack.append(rec)
        continue
    sel = sel[::PROBE_STRIDE][:MAX_PROBE_CROPS]
    probe = _unit(d['anon'][sel].mean(0))
    if probe is None:
        rec['status'] = 'no_probe'
        attack.append(rec)
        continue
    rec['n_probe'] = int(len(sel))
    _pc = d['anon'][sel]
    _pn = np.linalg.norm(_pc, axis=1, keepdims=True) + 1e-12
    rec['probe_consistency'] = float(np.mean((_pc / _pn) @ probe))

    candidates = id_index.get(r.split, {})
    keys = list(candidates.keys())
    # Gallery không chia sẻ qua các sequence khác (tránh rò rỉ split).
    pool = [k for k in keys if k[0] == r.sequence]
    true_key = (r.sequence, int(r.identity))
    if true_key not in pool:
        rec['status'] = 'no_gallery'
        attack.append(rec)
        continue
    if len(pool) > MAX_GALLERY_IDS:
        others = [k for k in pool if k != true_key]
        pick = rng.choice(len(others), size=min(len(others), MAX_GALLERY_IDS - 1), replace=False)
        pool = [true_key] + [others[i] for i in sorted(pick)]

    sims = {}
    for key in pool:
        entry = candidates[key]
        # Causal: chỉ lấy các embedding có frame <= cutoff
        # (identity thật dùng cutoff - gap để tránh sử dụng chính frame mẫu).
        if key == true_key:
            g = mean_upto(entry, cutoff - GALLERY_MIN_GAP)
        else:
            g = mean_upto(entry, cutoff)
        if g is not None:
            sims[key] = float(probe @ g)

    if true_key not in sims:
        rec['status'] = 'no_gallery'
        attack.append(rec)
        continue
    if len(sims) < MIN_GALLERY_IDS:
        rec['status'] = 'too_few_ids'
        attack.append(rec)
        continue

    order = sorted(sims.items(), key=lambda kv: -kv[1])
    top1_key, top1_sim = order[0]
    rec['n_gallery_ids'] = len(sims)
    rec['top1_identity'] = top1_key[1]
    rec['top1_sim'] = top1_sim
    rec['true_sim'] = sims[true_key]
    rec['rank_true'] = float([k for k, _ in order].index(true_key) + 1)
    rec['top1_correct'] = int(top1_key == true_key)
    attack.append(rec)

df_att = pd.DataFrame(attack)
df_att.to_csv(ATTACK_CSV, index=False)

print('\nattack_results.csv:', len(df_att), 'dong')
print('status distribution:')
print(df_att['status'].value_counts(dropna=False).to_string())
ok = df_att[df_att.status == 'ok']
if len(ok):
    print('top-1 đúng:', f'{ok.top1_correct.mean():.3f}',
          '| sim trung bình:', f'{ok.top1_sim.mean():.3f}')

(ARTIFACT / 'attack_meta.json').write_text(json.dumps({
    'backbone': reid_name, 'anon_method': ANON_METHOD,
    'anon_ksize': ANON_KSIZE, 'anon_block': ANON_PIXEL_BLOCK,
    'gallery_min_gap': GALLERY_MIN_GAP, 'cross_sequence': False,
    'gallery_max_per_id': GALLERY_MAX_PER_ID,
    'window': [WINDOW_SIZE, WINDOW_STRIDE], 'features': len(FEATURES),
}, indent=2), encoding='utf-8')

del model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ── CELL 10/13 · Chọn ngưỡng τ trên val (Youden J) ─────────
# Vào : attack_results.csv
# Ra  : tau.json, tau_sweep.csv, roc.png
#
# τ ở đây là ngưỡng cosine similarity để biến kết quả tấn công Re-ID thành
# nhãn nhị phân. Chọn trên val, không dùng test.

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

df_att = pd.read_csv(ATTACK_CSV)
val = df_att[(df_att.split == TAU_SELECT_SPLIT) & (df_att.status == 'ok')].copy()
print('mẫu val dùng chọn τ:', len(val))

tau_info = {'tau': 0.5, 'youden_j': None, 'auc': None,
            'n_val': int(len(val)), 'n_pos_val': int(val.top1_correct.fillna(0).sum()),
            'split': TAU_SELECT_SPLIT, 'margin': float(LABEL_TAU_MARGIN),
            'method': 'youden_j_val'}

if len(val) and val.top1_correct.nunique() > 1:
    y = val.top1_correct.astype(int).values
    score = val.top1_sim.values
    fpr, tpr, thr = roc_curve(y, score)
    j = tpr - fpr
    k = int(np.argmax(j))
    tau = float(np.clip(thr[k], 0.0, 1.0))
    if not np.isfinite(tau):
        tau = 0.5
    tau_info.update({'tau': tau, 'youden_j': float(j[k]),
                     'auc': float(roc_auc_score(y, score)),
                     'tpr': float(tpr[k]), 'fpr': float(fpr[k])})

    # Bảng sweep để audit lại ngưỡng sau này
    pd.DataFrame({'threshold': thr, 'tpr': tpr, 'fpr': fpr,
                  'youden': j}).to_csv(ARTIFACT / 'tau_sweep.csv', index=False)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, label=f'AUC = {tau_info["auc"]:.3f}')
    ax.plot([0, 1], [0, 1], '--', color='gray', lw=1)
    ax.scatter([fpr[k]], [tpr[k]], color='red', zorder=3,
               label=f'τ = {tau:.3f} (J = {j[k]:.3f})')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Re-ID attack ROC trên {TAU_SELECT_SPLIT}')
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(ARTIFACT / 'roc.png', dpi=140)
    plt.close(fig)
else:
    # Val thiếu một lớp (hoặc rỗng) -> không tính được Youden J.
    # Giữ τ = 0.5 mặc định (không chọn ngẫu nhiên) và ghi tau.json
    # để các cell sau (gán nhãn, train) vẫn chạy được.
    print('Cảnh báo: val thiếu lớp attack; dùng τ = 0.5 mặc định')

TAU_JSON.write_text(json.dumps(tau_info, indent=2, ensure_ascii=False), encoding='utf-8')
TAU = float(tau_info['tau'])
print(json.dumps(tau_info, indent=2, ensure_ascii=False))


In [ ]:
# ── CELL 11/13 · Gán nhãn + gộp track-level → risk_dataset.csv ──
# Vào : features_frame.csv, samples.csv, attack_results.csv, tau.json
# Ra  : risk_dataset.csv (25 đặc trưng + 4 chẩn đoán + label + trọng số)
#
# Sửa so với bản cũ:
#   1. Causal đúng nghĩa: chỉ dùng frame <= cutoff (= window_start). Không rò rỉ tương lai.
#   2. Không lấy trung bình toàn bộ lịch sử (dài vô hạn) nữa; mặc định lấy đúng
#      vector tại frame cuối cùng <= cutoff, khớp 1:1 với step5 runtime.
#      Có thể đổi FEATURE_AGG = 'window_mean' để lấy trung bình WINDOW_SIZE frame gần nhất.
#   3. Thêm label_margin / label_ambiguous / sample_weight để giảm ảnh hưởng nhãn nhiễu.

import math
from sklearn.metrics import roc_auc_score, average_precision_score

df_feat = pd.read_csv(FEATURES_CSV)
df_samples = pd.read_csv(SAMPLES_CSV)
df_att = pd.read_csv(ATTACK_CSV)
_tau_info = json.loads(TAU_JSON.read_text(encoding='utf-8'))
TAU = float(_tau_info['tau'])
TAU_MARGIN = float(_tau_info.get('margin', LABEL_TAU_MARGIN))

print(f'Kich thuoc dau vao: features={len(df_feat)}, samples={len(df_samples)}, attack={len(df_att)}')
print(f'Cac split co trong samples: {sorted(df_samples.split.unique().tolist()) if len(df_samples) else []}')

att_map = {r.sample_id: r for r in df_att.itertuples()}
feat_groups = {k: g.sort_values('frame')
               for k, g in df_feat.groupby(['split', 'sequence', 'track_id'])}


def aggregate_window(sub):
    # Gộp cả 4 đặc trưng chẩn đoán để cell metrics so sánh với đặc trưng giữ lại.
    cols = [f for f in list(FEATURES) + list(DIAGNOSTIC_FEATURES) if f in sub.columns]
    if FEATURE_AGG == 'last':
        row = sub.iloc[-1]
        return {f: float(row[f]) for f in cols}
    vals = sub[cols].to_numpy(np.float64)
    return {f: float(v) for f, v in zip(cols, vals.mean(0))}


records, skipped = [], Counter()
for r in df_samples.itertuples():
    g = feat_groups.get((r.split, r.sequence, int(r.track_id)))
    if g is None:
        skipped['no_features'] += 1
        continue

    cutoff = int(r.window_start)
    past = g[g.frame <= cutoff]
    if len(past) < MIN_WINDOW_FRAMES:
        skipped['short_window'] += 1
        continue
    hist = past.tail(WINDOW_SIZE) if WINDOW_SIZE and WINDOW_SIZE > 0 else past

    rec = {
        'sample_id': r.sample_id, 'split': r.split, 'sequence': r.sequence,
        'track_id': int(r.track_id), 'window_start': int(r.window_start),
        'window_end': int(r.window_end), 'cutoff': cutoff, 'identity': r.identity,
        'n_frames': int(len(past)), 'n_window': int(len(hist)),
    }
    rec.update(aggregate_window(hist))

    a = att_map.get(r.sample_id)
    if a is None or a.status != 'ok' or pd.isna(a.identity_true):
        rec['label'] = np.nan
        rec['label_reason'] = 'missing' if a is None else a.status
        rec['label_margin'] = np.nan
        rec['label_ambiguous'] = 0
        rec['sample_weight'] = 0.0
    else:
        top1_sim = float(a.top1_sim)
        true_sim = float(a.true_sim)
        correct = int(a.top1_correct) == 1
        rec['top1_identity'] = a.top1_identity
        rec['top1_sim'] = top1_sim
        rec['true_sim'] = true_sim
        rec['rank_true'] = a.rank_true
        rec['label_reason'] = 'attack'
        rec['label'] = 1.0 if (correct and top1_sim >= TAU) else 0.0
        # margin > 0: quyết định càng chắc chắn; margin < 0: sát biên
        signed = (top1_sim - TAU) if correct else (TAU - true_sim)
        rec['label_margin'] = float(signed)
        ambiguous = int(abs(signed) < TAU_MARGIN)
        rec['label_ambiguous'] = ambiguous
        if ambiguous:
            rec['sample_weight'] = 0.0
        else:
            rec['sample_weight'] = 1.0
    records.append(rec)

df_risk = pd.DataFrame(records)

# ---- Kiểm tra causal & toàn vẹn ----
if len(df_risk):
    assert (df_risk['cutoff'] <= df_risk['window_start']).all()
    assert df_risk['sample_id'].is_unique
    if 'n_window' in df_risk and WINDOW_SIZE and WINDOW_SIZE > 0:
        assert (df_risk['n_window'] <= WINDOW_SIZE).all()

df_risk.to_csv(RISK_CSV, index=False)

print('risk_dataset.csv:', len(df_risk), 'mẫu |', len(FEATURES), 'đặc trưng')
print('aggregation:', FEATURE_AGG, '| tau:', round(TAU, 4), '| margin:', TAU_MARGIN)
print('bỏ qua:', dict(skipped))
print('\nphân bố nhãn theo split:')
print(df_risk.groupby('split').label.agg(['count', 'sum', 'mean']).round(3).to_string())
print('\nnhãn tổng:', df_risk.label.value_counts(dropna=False).to_dict())
if len(df_risk) == 0:
    raise RuntimeError(
        'risk_dataset.csv rong (0 mau). Kiem tra: '
        f'features={len(df_feat)}, samples={len(df_samples)}, attack={len(df_att)}. '
        'Cell 9 (build_samples) hoac Cell 11 (attach label) khong tao duoc mau hop le.'
    )

# ---- Univariate AUC cho từng đặc trưng trên train (lọc đặc trưng yếu) ----
if len(df_risk) and df_risk['split'].eq('train').any():
    _tr = df_risk[df_risk['split'] == 'train']
    _y = _tr['label'].to_numpy(float)
    _uni_auc = {}
    _valid = ~np.isnan(_y)
    if _valid.sum() > 1 and len(np.unique(_y[_valid])) > 1:
        _y_valid = _y[_valid]
        print(f'Univariate AUC tren train: {_valid.sum()} mau co nhan (bo qua {len(_y) - _valid.sum()} mau NaN)')
        for _f in list(FEATURES) + list(DIAGNOSTIC_FEATURES):
            if _f in _tr.columns:
                _v = _tr[_f].to_numpy(float)[_valid]
                if np.unique(_v).size > 1:
                    _uni_auc[_f] = float(roc_auc_score(_y_valid, _v))
                else:
                    _uni_auc[_f] = None
        print('\nUnivariate AUC trên train:')
        for _f, _auc in sorted(_uni_auc.items(), key=lambda kv: (kv[1] is None, kv[1] or -1), reverse=True):
            _mark = '  ← KEU YẾU' if (_auc is not None and _auc < 0.55) else ''
            print('  %-22s AUC=%s%s' % (_f, 'NA' if _auc is None else '%.4f' % _auc, _mark))
        # Lưu vào artifact
        (ARTIFACT / 'univariate_auc.json').write_text(json.dumps(_uni_auc, indent=2, ensure_ascii=False), encoding='utf-8')
        print('univariate_auc.json ->', ARTIFACT / 'univariate_auc.json')
    else:
        print('Train chỉ có 1 lớp nhãn, bỏ qua univariate AUC.')
if 'label_ambiguous' in df_risk:
    amb = df_risk[df_risk.label.notna()]
    print('nhãn mờ (|margin| < %.3f): %d / %d = %.2f%%' % (
        TAU_MARGIN, int(amb.label_ambiguous.sum()), len(amb),
        100.0 * amb.label_ambiguous.mean()))
for f in list(FEATURES) + list(DIAGNOSTIC_FEATURES):
    assert f in df_risk.columns, f
print('\n25 cột đặc trưng model:', FEATURES)
print('4 cột chẩn đoán:', DIAGNOSTIC_FEATURES)


In [ ]:
# ── CELL 12/13 · Train XGBoost (sequence-disjoint + causal z-score + 2:1) ──
# Vào : risk_dataset.csv
# Ra  : risk_model.json, risk_model_calibrated.joblib, risk_oof.csv,
#       cv_summary.json, risk_tau.json, risk_norm_stats.json, risk_split_metrics.json
#
# Giao thức:
#   1. train / val / test phải sequence-disjoint (kiểm tra và raise nếu trùng).
#   2. Chuẩn hoá causal per-sequence z-score (Welford, min_n=NORM_MIN_SAMPLES)
#      RIÊNG từng split -> khớp SequenceStandardizer của step5 runtime.
#   3. GroupKFold theo sequence trên train để lấy OOF (không rò rỉ).
#   4. Undersample âm còn NEG_POS_RATIO:1 trong TỪNG fold train (giữ mọi dương).
#   5. Calibrate isotonic trên OOF (không dùng in-sample predictions như bản cũ).
#   6. Chọn tau vận hành trên val theo TARGET_FPR, lưu riêng risk_tau.json.
#   7. Test chỉ dùng để báo cáo cuối cùng, không tham gia train/chọn tau.

import joblib
import xgboost as xgb
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss, roc_curve)

SEED_TRAIN = int(SEED)


def ece_score(y_true, prob, bins=10):
    edges = np.linspace(0.0, 1.0, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (prob >= lo) & (prob < hi if hi < 1 else prob <= hi)
        if m.any():
            e += float(m.mean()) * abs(float(prob[m].mean()) - float(y_true[m].mean()))
    return e


def binary_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    if len(y) == 0:
        return {'n': 0}
    out = {'n': int(len(y)), 'positive_rate': float(y.mean())}
    if len(np.unique(y)) > 1:
        out.update({
            'auc': float(roc_auc_score(y, p)),
            'pr_auc': float(average_precision_score(y, p)),
            'brier': float(brier_score_loss(y, p)),
            'ece': float(ece_score(y, p)),
        })
    else:
        out.update({'auc': None, 'pr_auc': None,
                    'brier': float(brier_score_loss(y, p)) if len(y) else None,
                    'ece': None})
    return out


def causal_seq_zscore(frame):
    """z-score causal theo sequence: min_n vector đầu giữ nguyên, sau đó dùng
    mean/std của các vector TRƯỚC ĐÓ (Welford). Khớp step5.SequenceStandardizer."""
    X = frame[FEATURES].to_numpy(np.float64)
    out = np.empty_like(X)
    seq = frame['sequence'].to_numpy()
    time_col = 'cutoff' if 'cutoff' in frame.columns else 'window_start'
    order = np.lexsort((frame['track_id'].to_numpy(),
                        frame[time_col].to_numpy(),
                        seq))
    start, n = 0, len(order)
    while start < n:
        cur = seq[order[start]]
        end = start
        while end < n and seq[order[end]] == cur:
            end += 1
        mean = np.zeros(X.shape[1])
        m2 = np.zeros(X.shape[1])
        cnt = 0
        for k in range(start, end):
            i = order[k]
            x = X[i]
            cnt += 1
            if cnt <= NORM_MIN_SAMPLES:
                out[i] = x
            else:
                sd = np.sqrt(np.maximum(m2 / max(cnt - 1, 1), 1e-6))
                out[i] = (x - mean) / sd
            d = x - mean
            mean = mean + d / cnt
            m2 = m2 + d * (x - mean)
        start = end
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def undersample_negative(idx, y, groups, ratio, rng):
    """Giữ toàn bộ dương; lấy âm rải đều theo từng sequence để đạt ratio:1.
    Sequence không có dương vẫn được giữ một lượng âm đại diện (tránh mất domain)."""
    keep = []
    pos_per_seq = {}
    for g in np.unique(groups[idx]):
        gi = idx[groups[idx] == g]
        pos = gi[y[gi] == 1]
        neg = gi[y[gi] == 0]
        keep.extend(pos.tolist())
        if len(pos):
            pos_per_seq[g] = int(len(pos))
            n_keep = min(len(neg), int(round(len(pos) * ratio)))
            if n_keep > 0:
                sel = np.linspace(0, len(neg) - 1, n_keep).round().astype(int)
                keep.extend(neg[sel].tolist())
    if pos_per_seq:
        cap = max(1, int(round(ratio * float(np.median(list(pos_per_seq.values()))))))
        for g in np.unique(groups[idx]):
            if g in pos_per_seq:
                continue
            gi = idx[groups[idx] == g]
            neg = gi[y[gi] == 0]
            n_keep = min(len(neg), cap)
            if n_keep > 0:
                sel = np.linspace(0, len(neg) - 1, n_keep).round().astype(int)
                keep.extend(neg[sel].tolist())
    return np.array(sorted(set(int(v) for v in keep)), dtype=int)


def group_holdout(groups, frac=0.25, seed=0):
    """Tách một phần sequence để early stopping (không dùng val/test)."""
    u = np.unique(groups)
    if len(u) < 2:
        return np.arange(len(groups)), np.array([], dtype=int)
    rng = np.random.RandomState(seed)
    u = u.copy()
    rng.shuffle(u)
    k = max(1, int(round(len(u) * frac)))
    hold = set(u[:k].tolist())
    mask = np.array([g in hold for g in groups])
    return np.where(~mask)[0], np.where(mask)[0]


# ---------- 1. Đọc & kiểm tra dữ liệu ----------
df = pd.read_csv(RISK_CSV)
lab = df[df['label'].notna()].copy()
lab['label'] = lab['label'].astype(int)
lab['split'] = lab['split'].astype(str)
if 'sample_weight' not in lab.columns:
    lab['sample_weight'] = 1.0
lab['sample_weight'] = pd.to_numeric(lab['sample_weight'], errors='coerce').fillna(1.0)
if 'label_ambiguous' not in lab.columns:
    lab['label_ambiguous'] = 0
lab['label_ambiguous'] = lab['label_ambiguous'].fillna(0).astype(int)

# Chỉ loại nhãn mờ khỏi TRAIN; giữ nguyên val/test để đánh giá trung thực.
if DROP_AMBIGUOUS:
    amb = lab['label_ambiguous'].eq(1) & lab['split'].eq('train')
    if amb.any():
        print('Loại %d mẫu nhãn mờ khỏi train (vẫn giữ ở val/test)' % int(amb.sum()))
        lab = lab[~amb].copy()

parts = {s: lab[lab['split'] == s].copy() for s in ('train', 'val', 'test')}
# Fallback: nếu chưa có val, tách sequence khỏi train làm val
if len(parts['val']) == 0 and len(parts['train']) > 0:
    seqs = np.array(sorted(parts['train']['sequence'].unique()))
    rng = np.random.RandomState(SEED_TRAIN)
    rng.shuffle(seqs)
    n_hold = max(1, int(round(len(seqs) * 0.2)))
    hold = set(seqs[:n_hold].tolist())
    m = parts['train']['sequence'].isin(hold)
    parts['val'] = parts['train'][m].copy()
    parts['train'] = parts['train'][~m].copy()
    print('Tách %d sequence khỏi train làm val' % n_hold)
print('Phan bo split trong risk_dataset (co nhan):')
for _s in ('train', 'val', 'test'):
    _n = len(parts[_s])
    _pos = int(parts[_s]['label'].sum()) if _n else 0
    print(f'  {_s:<5}: {_n:>5} mau | duong: {_pos}')
if len(parts['train']) == 0:
    # Fallback: neu khong co split train, su dung toan bo du lieu co nhan lam train.
    if len(lab) > 0:
        print('CANH BAO: Khong co split train. Su dung toan bo du lieu co nhan lam train.')
        parts['train'] = lab.copy()
    else:
        raise RuntimeError(
            'Khong co mau co nhan nao trong risk_dataset.csv. '
            'Kiem tra Cell 11 (attack_results.csv) va Cell 9 (build_samples).'
        )

# Fallback: nếu chưa có val, tách sequence khỏi train.
if len(parts['val']) == 0:
    seqs = np.array(sorted(parts['train']['sequence'].unique()))
    rng = np.random.RandomState(SEED_TRAIN)
    rng.shuffle(seqs)
    n_hold = max(1, int(round(len(seqs) * 0.2)))
    hold = set(seqs[:n_hold].tolist())
    m = parts['train']['sequence'].isin(hold)
    parts['val'] = parts['train'][m].copy()
    parts['train'] = parts['train'][~m].copy()
    print('Không có val -> tách %d sequence khỏi train làm val' % n_hold)

# Kiểm tra sequence-disjoint
seq_sets = {s: set(parts[s]['sequence'].unique()) for s in parts if len(parts[s])}
for a, b in (('train', 'val'), ('train', 'test'), ('val', 'test')):
    if a in seq_sets and b in seq_sets:
        overlap = seq_sets[a] & seq_sets[b]
        if overlap:
            msg = 'Rò rỉ sequence giữa %s/%s: %s' % (a, b, sorted(overlap)[:5])
            if STRICT_SEQ_DISJOINT:
                raise RuntimeError(msg)
            print('CẢNH BÁO:', msg)

for s in ('train', 'val', 'test'):
    d = parts[s]
    if len(d):
        print('%-5s: %5d mẫu | dương %4d (%5.2f%%) | âm %5d | %2d sequence' % (
            s, len(d), int(d['label'].sum()), 100.0 * d['label'].mean(),
            int(len(d) - d['label'].sum()), d['sequence'].nunique()))

# Phân phối dương theo sequence (kiểm tra séquence quá ít/hô quá nhiều dương)
_tr_seq = parts['train'].groupby('sequence')['label'].agg(['count', 'sum'])
_tr_seq['pos_rate'] = (_tr_seq['sum'] / _tr_seq['count']).round(4)
print('\nPhân phối dương theo sequence (train):')
print(_tr_seq.sort_values('sum', ascending=False).to_string())
print('Số sequence có 0 dương:', int((_tr_seq['sum'] == 0).sum()), '/', len(_tr_seq))

# ---------- 2. Chuẩn hoá causal per-sequence, riêng từng split ----------
print('\nChuẩn hoá causal z-score theo sequence (min_n=%d)...' % NORM_MIN_SAMPLES)
X = {}
for s in ('train', 'val', 'test'):
    X[s] = causal_seq_zscore(parts[s]) if len(parts[s]) else np.zeros((0, len(FEATURES)), np.float32)

RISK_NORM_JSON.write_text(json.dumps({
    'method': 'causal per-sequence z-score (Welford)',
    'min_n': int(NORM_MIN_SAMPLES),
    'feature_agg': FEATURE_AGG,
    'features': list(FEATURES),
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'dropped_features': list(DROPPED_FEATURES),
    'applied_per_split': ['train', 'val', 'test'],
    'note': 'step5_integrated_system.SequenceStandardizer dùng cùng công thức',
}, indent=2, ensure_ascii=False), encoding='utf-8')

Xtr, ytr = X['train'], parts['train']['label'].to_numpy(int)
wtr = parts['train']['sample_weight'].to_numpy(float)
gtr = parts['train']['sequence'].to_numpy()

# Trọng số lớp từ dữ liệu train GỐC (không cân bằng) để XGBoost ưu tiên lớp dương hiếm
_pos_count = int((ytr == 1).sum())
_neg_count = int((ytr == 0).sum())
_pos_weight = float(min(_neg_count / max(_pos_count, 1), 5.0))  # giới hạn tối đa 5.0 để tránh overfit
print('Tỷ lệ âm:dương trong train (trước cân bằng): %.2f:1 | scale_pos_weight = %.2f (giới hạn 5.0)' % (
    _neg_count / max(_pos_count, 1), _pos_weight))

# ---------- 3. GroupKFold trên train -> OOF ----------
n_seq = len(np.unique(gtr))
n_splits = max(2, min(N_FOLDS, n_seq))
gkf = GroupKFold(n_splits=n_splits)
oof = np.full(len(ytr), np.nan)
folds, n_trees = [], []
rng_cv = np.random.RandomState(SEED_TRAIN)

for fold, (tr, va) in enumerate(gkf.split(Xtr, ytr, gtr)):
    tr_b = undersample_negative(tr, ytr, gtr, NEG_POS_RATIO, rng_cv)
    fit_rel, es_rel = group_holdout(gtr[tr_b], frac=0.25, seed=SEED_TRAIN + fold)
    fit_idx = tr_b[fit_rel]
    es_idx = tr_b[es_rel]

    params = dict(XGB_PARAMS)
    params['scale_pos_weight'] = _pos_weight  # Ưu tiên lớp dương hiếm
    use_es = (len(es_idx) > 0 and
              len(np.unique(ytr[fit_idx])) > 1 and
              len(np.unique(ytr[es_idx])) > 1)
    if not use_es:
        params.pop('early_stopping_rounds', None)

    model = xgb.XGBClassifier(**params)
    fit_kw = {'sample_weight': wtr[fit_idx], 'verbose': False}
    if use_es:
        fit_kw['eval_set'] = [(Xtr[es_idx], ytr[es_idx])]
        fit_kw['sample_weight_eval_set'] = [wtr[es_idx]]
    model.fit(Xtr[fit_idx], ytr[fit_idx], **fit_kw)

    oof[va] = model.predict_proba(Xtr[va])[:, 1]
    if use_es:
        n_trees.append(int(model.best_iteration) + 1)
    else:
        n_trees.append(int(params.get('n_estimators', XGB_PARAMS['n_estimators'])))

    fold_auc = (float(roc_auc_score(ytr[va], oof[va]))
                if len(np.unique(ytr[va])) > 1 else None)
    fold_ap = (float(average_precision_score(ytr[va], oof[va]))
               if len(np.unique(ytr[va])) > 1 else None)
    folds.append({
        'fold': fold,
        'n_train_fit': int(len(fit_idx)),
        'n_early_stop': int(len(es_idx)),
        'n_val': int(len(va)),
        'pos_val': int(ytr[va].sum()),
        'neg_pos_ratio': float(len(fit_idx[ ytr[fit_idx] == 0 ]) / max(1, int(ytr[fit_idx].sum()))),
        'auc': fold_auc,
        'pr_auc': fold_ap,
        'n_trees': int(n_trees[-1]),
    })
    print('  fold %d: n_fit=%d n_es=%d n_val=%d | AUC=%s PR-AUC=%s | trees=%d' % (
        fold, len(fit_idx), len(es_idx), len(va),
        'NA' if fold_auc is None else '%.4f' % fold_auc,
        'NA' if fold_ap is None else '%.4f' % fold_ap,
        n_trees[-1]))

oof_mask = np.isfinite(oof)
if oof_mask.sum() == 0 or len(np.unique(ytr[oof_mask])) < 2:
    raise RuntimeError('OOF không hợp lệ (thiếu lớp) - kiểm tra lại dữ liệu train')
oof_auc = float(roc_auc_score(ytr[oof_mask], oof[oof_mask]))
oof_ap = float(average_precision_score(ytr[oof_mask], oof[oof_mask]))
print('\nOOF (sequence-disjoint): AUC=%.4f | PR-AUC=%.4f' % (oof_auc, oof_ap))

# ---------- 4. Model cuối trên train ----------
n_est_final = int(np.clip(np.median(n_trees) if n_trees else 300, 50,
                          int(XGB_PARAMS['n_estimators'])))
params_final = {k: v for k, v in XGB_PARAMS.items() if k != 'early_stopping_rounds'}
params_final['scale_pos_weight'] = _pos_weight  # Ưu tiên lớp dương hiếm
params_final['n_estimators'] = n_est_final

train_idx = undersample_negative(np.arange(len(ytr)), ytr, gtr, NEG_POS_RATIO, rng_cv)
final_model = xgb.XGBClassifier(**params_final)
final_model.fit(Xtr[train_idx], ytr[train_idx],
                sample_weight=wtr[train_idx], verbose=False)
print('Model cuối: n_estimators=%d | train_fit=%d | neg:pos=%.2f' % (
    n_est_final, len(train_idx),
    float(len(train_idx[ytr[train_idx] == 0]) / max(1, int(ytr[train_idx].sum())))))

# ---------- 5. Calibration isotonic trên OOF ----------
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof[oof_mask], ytr[oof_mask])
_iso_x = np.unique(np.asarray(oof[oof_mask], dtype=float))
_iso_y = np.clip(iso.predict(_iso_x), 0.0, 1.0)


def calibrate(prob):
    return np.clip(np.interp(np.asarray(prob, dtype=float), _iso_x, _iso_y), 0.0, 1.0)


# ---------- 6. Đánh giá train/val/test ----------
def evaluate_split(name):
    d = parts[name]
    if len(d) == 0:
        return None
    raw = final_model.predict_proba(X[name])[:, 1]
    cal = calibrate(raw)
    res = {'raw': binary_metrics(d['label'].to_numpy(int), raw),
           'calibrated': binary_metrics(d['label'].to_numpy(int), cal)}
    print('  %-5s raw: AUC=%s PR-AUC=%s | cal: AUC=%s PR-AUC=%s' % (
        name,
        'NA' if res['raw']['auc'] is None else '%.4f' % res['raw']['auc'],
        'NA' if res['raw']['pr_auc'] is None else '%.4f' % res['raw']['pr_auc'],
        'NA' if res['calibrated']['auc'] is None else '%.4f' % res['calibrated']['auc'],
        'NA' if res['calibrated']['pr_auc'] is None else '%.4f' % res['calibrated']['pr_auc']))
    return res


print('\n=== Đánh giá theo split ===')
split_metrics = {s: evaluate_split(s) for s in ('train', 'val', 'test')}

# ---------- 7. Chọn tau trên val ----------
tau = float(RISK_THRESHOLD)
tau_info = {'tau': tau, 'method': 'fixed', 'target_fpr': float(TARGET_FPR)}
if len(parts['val']):
    p_val = calibrate(final_model.predict_proba(X['val'])[:, 1])
    y_val = parts['val']['label'].to_numpy(int)
    _n_pos_val = int((y_val == 1).sum())
    # Val quá ít dương (<50) -> chọn τ theo OOF (toàn bộ train sequence-disjoint) ổn định hơn
    if _n_pos_val < 50:
        print('Cảnh báo: val chỉ có %d mẫu dương -> chọn τ theo OOF thay vì val' % _n_pos_val)
        _oof_mask = np.isfinite(oof)
        _y_oof = ytr[_oof_mask]
        _p_oof = oof[_oof_mask]
        _neg_oof = _p_oof[_y_oof == 0]
        if len(_neg_oof) and len(np.unique(_y_oof)) > 1:
            tau = float(np.quantile(_neg_oof, 1.0 - TARGET_FPR))
            # Tính Youden trên OOF
            _fprs, _tprs, _thrs = roc_curve(_y_oof, _p_oof)
            _k = int(np.argmax(_tprs - _fprs))
            tau_youden_oof = float(_thrs[_k])
            if not np.isfinite(tau_youden_oof):
                tau_youden_oof = tau
        else:
            tau = float(RISK_THRESHOLD)
            tau_youden_oof = tau
    else:
        neg = p_val[y_val == 0]
        if len(neg):
            tau = float(np.quantile(neg, 1.0 - TARGET_FPR))
            _fprs, _tprs, _thrs = roc_curve(y_val, p_val)
            _k = int(np.argmax(_tprs - _fprs))
            tau_youden_oof = float(_thrs[_k])
            if not np.isfinite(tau_youden_oof):
                tau_youden_oof = tau
        else:
            tau = float(RISK_THRESHOLD)
            tau_youden_oof = tau
    print('Chọn τ theo OOF/val: tau=%.4f | target_fpr=%.0f%%' % (tau, 100 * TARGET_FPR))
    picked = (p_val >= tau).astype(int)
    tpr = float(picked[y_val == 1].mean()) if (y_val == 1).any() else 0.0
    fpr = float(picked[y_val == 0].mean()) if (y_val == 0).any() else 0.0
    if len(np.unique(y_val)) > 1:
        fprs, tprs, thrs = roc_curve(y_val, p_val)
        k = int(np.argmax(tprs - fprs))
        tau_youden = float(thrs[k])
        youden = float(tprs[k] - fprs[k])
        if not np.isfinite(tau_youden):
            tau_youden = tau
    else:
        tau_youden, youden = tau, None
    tau_info = {
        'tau': tau,
        'method': 'target_fpr_val',
        'target_fpr': float(TARGET_FPR),
        'achieved_tpr': tpr,
        'achieved_fpr': fpr,
        'tau_youden': tau_youden,
        'youden_j': youden,
        'n_val': int(len(y_val)),
        'n_positive': int(y_val.sum()),
        'evaluation': 'sequence-disjoint hold-out (val split)',
    }
    print('\nTau (target FPR=%.0f%%): %.4f | TPR=%.4f FPR=%.4f | Youden=%.4f' % (
        100 * TARGET_FPR, tau, tpr, fpr, tau_youden))
RISK_TAU_JSON.write_text(json.dumps(tau_info, indent=2, ensure_ascii=False), encoding='utf-8')

# confusion matrix tại tau cho val/test
for s in ('val', 'test'):
    if len(parts[s]) and split_metrics[s] is not None:
        p = calibrate(final_model.predict_proba(X[s])[:, 1])
        y = parts[s]['label'].to_numpy(int)
        yh = (p >= tau).astype(int)
        split_metrics[s]['confusion_at_tau'] = [
            [int(((y == 0) & (yh == 0)).sum()), int(((y == 0) & (yh == 1)).sum())],
            [int(((y == 1) & (yh == 0)).sum()), int(((y == 1) & (yh == 1)).sum())],
        ]

# ---------- 8. Lưu model + payload ----------
final_model.save_model(str(RISK_MODEL_JSON))
if 'save_model_metadata' in globals():
    save_model_metadata(RISK_MODEL_JSON, FEATURES)
print('risk_model.json ->', RISK_MODEL_JSON)

val_raw = (binary_metrics(parts['val']['label'].to_numpy(int),
                          final_model.predict_proba(X['val'])[:, 1])
           if len(parts['val']) else {})
val_cal = split_metrics['val']['calibrated'] if split_metrics['val'] else {}

payload = {
    'model': final_model,
    'calibrator': {'X_thresholds_': _iso_x, 'y_values_': _iso_y},
    'features': list(FEATURES),
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'feature_groups': FEATURE_GROUPS,
    'dropped_features': list(DROPPED_FEATURES),
    'normalization': {'method': 'causal per-sequence z-score (Welford)',
                      'min_n': int(NORM_MIN_SAMPLES),
                      'feature_agg': FEATURE_AGG},
    'tau': tau,
    'ece_raw': val_raw.get('ece'),
    'ece_calibrated': val_cal.get('ece'),
    'brier_raw': val_raw.get('brier'),
    'brier_calibrated': val_cal.get('brier'),
}
joblib.dump(payload, ARTIFACT / 'risk_model_calibrated.joblib')
print('risk_model_calibrated.joblib ->', ARTIFACT / 'risk_model_calibrated.joblib')

# ---------- 9. Lưu OOF + summary ----------
oof_df = pd.DataFrame({
    'sample_id': parts['train']['sample_id'].to_numpy(),
    'split': 'train_oof',
    'sequence': parts['train']['sequence'].to_numpy(),
    'y_true': ytr,
    'y_prob': oof,
})
extra = []
for s in ('val', 'test'):
    if len(parts[s]):
        extra.append(pd.DataFrame({
            'sample_id': parts[s]['sample_id'].to_numpy(),
            'split': s,
            'sequence': parts[s]['sequence'].to_numpy(),
            'y_true': parts[s]['label'].to_numpy(int),
            'y_prob': calibrate(final_model.predict_proba(X[s])[:, 1]),
        }))
pd.concat([oof_df] + extra, ignore_index=True).to_csv(RISK_OOF_CSV, index=False)
print('risk_oof.csv ->', RISK_OOF_CSV)

cv_summary = {
    'source_csv': str(RISK_CSV),
    'n_samples': int(len(lab)),
    'n_train': int(len(parts['train'])),
    'n_val': int(len(parts['val'])),
    'n_test': int(len(parts['test'])),
    'positives': {'train': int(ytr.sum()),
                  'val': int(parts['val']['label'].sum()) if len(parts['val']) else 0,
                  'test': int(parts['test']['label'].sum()) if len(parts['test']) else 0},
    'neg_pos_ratio': float(NEG_POS_RATIO),
    'feature_agg': FEATURE_AGG,
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'features': list(FEATURES),
    'dropped_features': list(DROPPED_FEATURES),
    'feature_groups': FEATURE_GROUPS,
    'normalization': 'causal per-sequence z-score (Welford, min_n=%d), per split' % NORM_MIN_SAMPLES,
    'protocol': 'GroupKFold(sequence) OOF on train; early stopping on inner sequence holdout; val for tau; test untouched',
    'params': {k: v for k, v in params_final.items() if k != 'n_jobs'},
    'n_trees_per_fold': [int(v) for v in n_trees],
    'n_estimators_final': int(n_est_final),
    'n_splits': int(n_splits),
    'folds': folds,
    'oof_auc': oof_auc,
    'oof_pr_auc': oof_ap,
    'calibration': {'method': 'isotonic on OOF', 'n_oof': int(oof_mask.sum())},
    'tau': tau_info,
    'split_metrics': split_metrics,
}
(ARTIFACT / 'cv_summary.json').write_text(
    json.dumps(cv_summary, indent=2, ensure_ascii=False), encoding='utf-8')
RISK_SPLIT_METRICS_JSON.write_text(
    json.dumps(split_metrics, indent=2, ensure_ascii=False), encoding='utf-8')
print('cv_summary.json / risk_split_metrics.json đã lưu')
print(json.dumps({'oof_auc': oof_auc, 'oof_pr_auc': oof_ap, 'tau': tau},
                 indent=2, ensure_ascii=False))


In [ ]:
# ── CELL 13/13 · Đo hiệu năng + chẩn đoán đặc trưng + SHAP + export ──
# Vào : risk_model.json, risk_model_calibrated.joblib, risk_oof.csv,
#       risk_dataset.csv, risk_tau.json
# Ra  : risk_metrics.json, calib.png, importance.png, shap.png,
#       feature_importance.csv, zip
#
# Sửa so với bản cũ:
#   - Không còn báo "OOF toàn bộ" trộn train với test.
#   - Báo cáo riêng val (chọn tau) và test (hold-out sequence-disjoint).
#   - Calibration được fit trên OOF của train, không fit trên chính tập train.
#   - Thêm bootstrap CI, per-sequence AUC, SMD train-vs-test, baseline.

from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, brier_score_loss,
                             confusion_matrix)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
import joblib
import matplotlib.pyplot as plt

df = pd.read_csv(RISK_CSV)
lab = df[df['label'].notna()].copy()
lab['label'] = lab['label'].astype(int)
lab['split'] = lab['split'].astype(str)

parts = {s: lab[lab['split'] == s].copy() for s in ('train', 'val', 'test')}
# Fallback: nếu chưa có val, tách sequence khỏi train làm val
if len(parts['val']) == 0 and len(parts['train']) > 0:
    seqs = np.array(sorted(parts['train']['sequence'].unique()))
    rng = np.random.RandomState(SEED_TRAIN)
    rng.shuffle(seqs)
    n_hold = max(1, int(round(len(seqs) * 0.2)))
    hold = set(seqs[:n_hold].tolist())
    m = parts['train']['sequence'].isin(hold)
    parts['val'] = parts['train'][m].copy()
    parts['train'] = parts['train'][~m].copy()
    print('Tách %d sequence khỏi train làm val' % n_hold)
# Đồng bộ với cell train: mẫu nhãn mờ không tham gia train.
if DROP_AMBIGUOUS and 'label_ambiguous' in parts['train'].columns:
    parts['train'] = parts['train'][
        parts['train']['label_ambiguous'].fillna(0).astype(int) == 0].copy()


def _causal_seq_zscore(frame):
    X = frame[FEATURES].to_numpy(np.float64)
    out = np.empty_like(X)
    seq = frame['sequence'].to_numpy()
    time_col = 'cutoff' if 'cutoff' in frame.columns else 'window_start'
    order = np.lexsort((frame['track_id'].to_numpy(), frame[time_col].to_numpy(), seq))
    start, n = 0, len(order)
    while start < n:
        cur = seq[order[start]]
        end = start
        while end < n and seq[order[end]] == cur:
            end += 1
        mean = np.zeros(X.shape[1])
        m2 = np.zeros(X.shape[1])
        cnt = 0
        for k in range(start, end):
            i = order[k]
            x = X[i]
            cnt += 1
            if cnt <= NORM_MIN_SAMPLES:
                out[i] = x
            else:
                sd = np.sqrt(np.maximum(m2 / max(cnt - 1, 1), 1e-6))
                out[i] = (x - mean) / sd
            d = x - mean
            mean = mean + d / cnt
            m2 = m2 + d * (x - mean)
        start = end
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def _ece(y_true, prob, bins=10):
    edges = np.linspace(0.0, 1.0, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (prob >= lo) & (prob < hi if hi < 1 else prob <= hi)
        if m.any():
            e += float(m.mean()) * abs(float(prob[m].mean()) - float(y_true[m].mean()))
    return float(e)


def _metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    out = {'n': int(len(y)), 'positive_rate': float(y.mean()) if len(y) else None,
           'auc': None, 'pr_auc': None, 'brier': None, 'ece': None}
    if len(y):
        if len(np.unique(y)) > 1:
            out['auc'] = float(roc_auc_score(y, p))
            out['pr_auc'] = float(average_precision_score(y, p))
            out['ece'] = _ece(y, p)
        p_clipped = np.clip(p, 0.0, 1.0)  # brier_score_loss requires prob in [0,1]
        out['brier'] = float(brier_score_loss(y, p_clipped))
    return out


def _boot_auc(y, p, n_boot=1000, seed=0):
    y = np.asarray(y)
    p = np.asarray(p)
    if len(np.unique(y)) < 2:
        return None
    rng = np.random.RandomState(seed)
    vals = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], p[idx]))
    if not vals:
        return None
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]


cal = joblib.load(ARTIFACT / 'risk_model_calibrated.joblib')
model = cal['model'] if isinstance(cal, dict) and 'model' in cal else cal
cx = np.asarray(cal['calibrator']['X_thresholds_'], dtype=float) if isinstance(cal, dict) else None
cy = np.asarray(cal['calibrator']['y_values_'], dtype=float) if isinstance(cal, dict) else None


def _calibrate(p):
    if cx is None:
        return np.asarray(p, dtype=float)
    return np.clip(np.interp(np.asarray(p, dtype=float), cx, cy), 0.0, 1.0)


tau_info = json.loads(RISK_TAU_JSON.read_text(encoding='utf-8')) if RISK_TAU_JSON.is_file() else {}
tau = float(tau_info.get('tau', RISK_THRESHOLD))

X, Y, P_RAW, P_CAL = {}, {}, {}, {}
for s in ('train', 'val', 'test'):
    if len(parts[s]) == 0:
        continue
    X[s] = _causal_seq_zscore(parts[s])
    Y[s] = parts[s]['label'].to_numpy(int)
    P_RAW[s] = model.predict_proba(X[s])[:, 1]
    P_CAL[s] = _calibrate(P_RAW[s])

report = {
    'evaluation_protocol': ('sequence-disjoint: GroupKFold OOF + inner early-stop on train; '
                            'val cho tau; test là hold-out không tham gia train/chọn tau'),
    'n_train': int(len(parts['train'])),
    'n_val': int(len(parts['val'])),
    'n_test': int(len(parts['test'])),
    'threshold': {'tau': tau, 'method': tau_info.get('method'), 'target_fpr': TARGET_FPR},
    'normalization': 'causal per-sequence z-score (Welford, min_n=%d), per split' % NORM_MIN_SAMPLES,
    'splits': {},
    'per_sequence_test': {},
    'bootstrap_test_auc_ci95': None,
    'baselines': {},
    'feature_shift_train_vs_test': {},
    'univariate_auc_train': {},
}

for s in ('train', 'val', 'test'):
    if s not in X:
        continue
    raw_m = _metrics(Y[s], P_RAW[s])
    cal_m = _metrics(Y[s], P_CAL[s])
    yh = (P_CAL[s] >= tau).astype(int)
    entry = {'raw': raw_m, 'calibrated': cal_m,
             'confusion_at_tau': confusion_matrix(Y[s], yh, labels=[0, 1]).tolist(),
             'at_tau': {
                 'f1': float(f1_score(Y[s], yh, zero_division=0)),
                 'precision': float(precision_score(Y[s], yh, zero_division=0)),
                 'recall': float(recall_score(Y[s], yh, zero_division=0)),
             }}
    report['splits'][s] = entry

# per-sequence trên test
if 'test' in X:
    for seq in sorted(parts['test']['sequence'].unique()):
        m = parts['test']['sequence'].to_numpy() == seq
        yy, pp = Y['test'][m], P_CAL['test'][m]
        report['per_sequence_test'][seq] = _metrics(yy, pp)
    report['bootstrap_test_auc_ci95'] = _boot_auc(Y['test'], P_CAL['test'])

# baseline trên test (fit chỉ trên train)
if 'train' in X and 'test' in X:
    prior = float(Y['train'].mean())
    report['baselines']['prior'] = {
        'auc': 0.5,
        'brier_test': float(brier_score_loss(Y['test'], np.full(len(Y['test']), prior))),
        'pr_auc_test': float(Y['test'].mean()),
    }
    lr = LogisticRegression(max_iter=3000, class_weight='balanced')
    lr.fit(X['train'], Y['train'])
    lp = lr.predict_proba(X['test'])[:, 1]
    report['baselines']['logreg_train_to_test'] = _metrics(Y['test'], lp)
    conf_idx = FEATURES.index('conf')
    report['baselines']['conf_threshold_test'] = _metrics(Y['test'], X['test'][:, conf_idx])

# feature shift (SMD) train vs test + univariate AUC trên train
if 'train' in X and 'test' in X:
    a_df = parts['train'][FEATURES].to_numpy(float)
    b_df = parts['test'][FEATURES].to_numpy(float)
    for i, f in enumerate(FEATURES):
        a, b = a_df[:, i], b_df[:, i]
        sd = np.sqrt((a.var() + b.var()) / 2.0) + 1e-12
        report['feature_shift_train_vs_test'][f] = float(abs(a.mean() - b.mean()) / sd)
        if len(np.unique(Y['train'])) > 1:
            report['univariate_auc_train'][f] = float(roc_auc_score(Y['train'], a))
    # Đặc trưng bị loại: vẫn đo univariate AUC để chứng minh quyết định loại bỏ.
    for f in DIAGNOSTIC_FEATURES:
        if f in parts['train'].columns and len(np.unique(Y['train'])) > 1:
            report['univariate_auc_train'][f] = float(
                roc_auc_score(Y['train'], parts['train'][f].to_numpy(float)))

# ---- giải thích lựa chọn đặc trưng + permutation importance ----
report['feature_selection'] = {
    'schema_version': FEATURE_SCHEMA_VERSION,
    'retained': list(FEATURES),
    'dropped': list(DROPPED_FEATURES),
    'drop_rationale': FEATURE_DROP_RATIONALE,
    'feature_groups': FEATURE_GROUPS,
}
report['permutation_importance'] = {}
report['feature_group_importance'] = {}
_pi_split = ('test' if 'test' in X else ('val' if 'val' in X else ('train' if 'train' in X else None)))
if _pi_split:
    try:
        from sklearn.inspection import permutation_importance
        n_rep = 5 if len(Y[_pi_split]) >= 100 else 3
        pi = permutation_importance(
            model, X[_pi_split], Y[_pi_split], scoring='roc_auc',
            n_repeats=n_rep, random_state=SEED, n_jobs=1,
        )
        report['permutation_importance'] = {
            f: {'mean': float(pi.importances_mean[i]),
                'std': float(pi.importances_std[i])}
            for i, f in enumerate(FEATURES)
        }
        group_imp = {g: 0.0 for g in FEATURE_GROUPS}
        for i, f in enumerate(FEATURES):
            g = next((k for k, v in FEATURE_GROUPS.items() if f in v), 'other')
            group_imp[g] = group_imp.get(g, 0.0) + max(0.0, float(pi.importances_mean[i]))
        report['feature_group_importance'] = group_imp
        report['permutation_importance_split'] = _pi_split
        print('Permutation importance (%s):' % _pi_split)
        for f, v in sorted(report['permutation_importance'].items(),
                           key=lambda kv: kv[1]['mean'], reverse=True)[:10]:
            print('  %-22s %.4f +/- %.4f' % (f, v['mean'], v['std']))
    except Exception as e:
        print('Permutation importance lỗi:', str(e)[:160])
        report['permutation_importance_error'] = str(e)[:300]

# ---- top-level tương thích các script cũ (dùng test nếu có, ngược lại val) ----
primary = 'test' if 'test' in P_CAL else ('val' if 'val' in P_CAL else None)
if primary:
    pm = report['splits'][primary]
    report.update({
        'primary_split': primary,
        'auc': pm['calibrated']['auc'],
        'pr_auc': pm['calibrated']['pr_auc'],
        'brier': pm['calibrated']['brier'],
        'ece': pm['calibrated']['ece'],
        'f1': pm['at_tau']['f1'],
        'precision': pm['at_tau']['precision'],
        'recall': pm['at_tau']['recall'],
        'confusion_matrix': pm['confusion_at_tau'],
        'calibration_used': True,
    })

# ---- calibration curve trên primary ----
if primary:
    try:
        frac_pos, mean_pred = calibration_curve(Y[primary], P_CAL[primary],
                                                n_bins=10, strategy='quantile')
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.plot(mean_pred, frac_pos, 'o-', label='XGBoost (isotonic, OOF)')
        ax.plot([0, 1], [0, 1], '--', color='gray', lw=1)
        ax.set_xlabel('Xác suất dự đoán')
        ax.set_ylabel('Tỷ lệ dương thực tế')
        ax.set_title('Calibration (%s) | Brier=%.4f | ECE=%.4f' % (
            primary, report['brier'], report['ece']))
        ax.legend()
        fig.tight_layout()
        fig.savefig(ARTIFACT / 'calib.png', dpi=140)
        plt.close(fig)
    except Exception as e:
        print('Calibration plot lỗi:', str(e)[:160])

# ---- feature importance ----
import xgboost as xgb
model_x = xgb.XGBClassifier()
model_x.load_model(str(RISK_MODEL_JSON))
gain_map = model_x.get_booster().get_score(importance_type='gain')
imp = pd.DataFrame({
    'feature': FEATURES,
    'gain': [float(gain_map.get('f%d' % i, gain_map.get(f, 0.0))) for i, f in enumerate(FEATURES)],
})
imp = imp.sort_values('gain', ascending=False)
imp.to_csv(ARTIFACT / 'feature_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(imp.feature[::-1], imp.gain[::-1], color='steelblue')
ax.set_xlabel('Gain')
ax.set_title('XGBoost feature importance')
fig.tight_layout()
fig.savefig(ARTIFACT / 'importance.png', dpi=140)
plt.close(fig)

# ---- SHAP (tuỳ chọn) ----
try:
    import shap
    explainer = shap.TreeExplainer(model_x)
    X_all = np.concatenate([X[s] for s in X], axis=0)
    sample_n = min(500, len(X_all))
    sv = explainer.shap_values(X_all[:sample_n])
    fig = plt.figure(figsize=(8, 6))
    shap.summary_plot(sv, X_all[:sample_n], feature_names=FEATURES, show=False)
    fig.tight_layout()
    fig.savefig(ARTIFACT / 'shap.png', dpi=140, bbox_inches='tight')
    plt.close(fig)
    report['shap'] = 'shap.png'
except Exception as e:
    print('SHAP không khả dụng:', str(e)[:160])
    report['shap'] = None

# ---- export detector ----
if EXPORT_ONNX and BEST_MODEL is not None and BEST_MODEL.suffix.lower() == '.pt' and BEST_MODEL.is_file():
    try:
        _work_model = WORK / 'onnx_export' / BEST_MODEL.name
        _work_model.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(BEST_MODEL, _work_model)
        exp = YOLO(str(_work_model))
        onnx_path = Path(exp.export(format='onnx', imgsz=IMGSZ_USED, opset=ONNX_OPSET,
                                    dynamic=False, simplify=True, half=False))
        shutil.copy2(onnx_path, ARTIFACT / onnx_path.name)
        print('ONNX ->', ARTIFACT / onnx_path.name)
        del exp
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print('Export ONNX lỗi:', str(e)[:160])
elif EXPORT_ONNX:
    print('Bỏ qua export ONNX: không tìm thấy file .pt nguồn (BEST_MODEL =', BEST_MODEL, ')')

if BEST_MODEL is not None and BEST_MODEL.suffix.lower() == '.pt' and BEST_MODEL.is_file():
    shutil.copy2(BEST_MODEL, ARTIFACT / 'yolo26s.pt')
if DATA_YAML is not None and DATA_YAML.is_file():
    shutil.copy2(DATA_YAML, ARTIFACT / 'data_person.yaml')

RISK_METRICS_JSON.write_text(json.dumps(report, indent=2, ensure_ascii=False),
                             encoding='utf-8')
print(json.dumps({k: v for k, v in report.items()
                  if k not in ('splits', 'per_sequence_test', 'feature_shift_train_vs_test',
                               'univariate_auc_train')},
                 indent=2, ensure_ascii=False))

zip_path = create_results_zip() if 'create_results_zip' in globals() else WORK / 'visdrone_privacy_results.zip'
print('\nfile ZIP tổng ->', zip_path, '(%0.1f MiB)' % (Path(zip_path).stat().st_size / 2**20))
print('artifact:', sorted(p.name for p in ARTIFACT.iterdir()))


In [ ]:
# CELL 14/14 (bo sung) - Dong goi toan bo du an thanh ZIP de tai xuong
import zipfile
from pathlib import Path
from datetime import datetime
from IPython.display import display, FileLink

PROJECT_ROOT = Path.cwd()
ZIP_NAME = 'visdrone_privacy_project_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.zip'
ZIP_PATH = PROJECT_ROOT / ZIP_NAME

SKIP_DIRS = {'.git', '__pycache__', '.ipynb_checkpoints', '.cache', 'node_modules'}
SKIP_SUFFIXES = {'.pt', '.pth', '.zip', '.rar', '.7z', '.onnx', '.mp4', '.avi', '.wav'}
SKIP_NAMES = {'.DS_Store'}

count = 0
skipped_large = 0
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in PROJECT_ROOT.rglob('*'):
        rel = p.relative_to(PROJECT_ROOT)
        if any(part in SKIP_DIRS for part in rel.parts):
            continue
        if p.is_dir():
            continue
        if p.name == ZIP_NAME or p.name in SKIP_NAMES or p.suffix.lower() in SKIP_SUFFIXES:
            continue
        try:
            if p.stat().st_size > 200 * 1024 * 1024:
                skipped_large += 1
                continue
        except OSError:
            continue
        zf.write(p, rel)
        count += 1

size_mb = ZIP_PATH.stat().st_size / 2**20
print('Da tao ZIP:', ZIP_PATH)
print('So tep:', count, '| Kich thuoc: %.1f MiB' % size_mb)
if skipped_large:
    print('Bo qua', skipped_large, 'tep lon hon 200 MiB.')
display(FileLink(str(ZIP_PATH)))
